In [ ]:
## knockdown code

In [ ]:
#aggreagation            env= dp_morning_repro

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
          "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
          "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

# Paths for the two separate outputs
OUTPUT_CSV_MEDIAN = os.path.join(PROJECT_ROOT,"files", "aggregated_wells_median.csv")
OUTPUT_CSV_STD = os.path.join(PROJECT_ROOT,"files", "aggregated_wells_std.csv")

CELL_COUNT_THRESHOLD = 0  
TREATMENT_COL = "Treatment"

all_plates_median = []
all_plates_std = []
all_cell_counts = [] 

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT, "features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT, "metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    # --- AGGREGATION & THRESHOLDING STEP ---
    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_cell_count = all_cells_in_well.shape[0]
        all_cell_counts.append(well_cell_count)

        if well_cell_count >= CELL_COUNT_THRESHOLD:
            # Calculate both Median and Std Dev
            well_median = np.median(all_cells_in_well, axis=0)
            well_std = np.std(all_cells_in_well, axis=0)
            
            base_info = {
                "Plate": plate_id, 
                "Well_ID": well_id, 
                "Treatment": well_to_treatment[well_id],
                "Cell_Count": well_cell_count
            }
            
            # Create rows for both dataframes
            row_median = base_info.copy()
            row_std = base_info.copy()
            
            for idx in range(len(well_median)):
                row_median[idx] = well_median[idx]
                row_std[idx] = well_std[idx]
                
            all_plates_median.append(row_median)
            all_plates_std.append(row_std)

# Convert to DataFrames
df_median = pd.DataFrame(all_plates_median)
df_std = pd.DataFrame(all_plates_std)

# Helper function to reorder
def reorder_cols(df):
    meta_cols = ["Plate", "Well_ID", "Treatment", "Cell_Count"]
    feat_cols = [c for c in df.columns if c not in meta_cols]
    return df[meta_cols + feat_cols]

df_median = reorder_cols(df_median)
df_std = reorder_cols(df_std)

print(f"\nAggregation complete.")

# ==========================================
# 2. VISUALIZATION: CELL COUNT HISTOGRAM
# ==========================================
counts = np.array(all_cell_counts)
c_mean = np.mean(counts)
c_median = np.median(counts)
c_std = np.std(counts)

plt.figure(figsize=(10, 6))
plt.hist(counts, bins=50, color='skyblue', edgecolor='black', alpha=0.7)

# Create stats text string
stats_text = f'Mean: {c_mean:.2f}\nMedian: {c_median:.2f}\nStd Dev: {c_std:.2f}'
# Place text box in the plot
plt.gca().text(0.95, 0.95, stats_text, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

plt.title('Distribution of Cell Counts per Well (All Plates)')
plt.xlabel('Number of Cells')
plt.ylabel('Frequency (Wells)')
plt.grid(axis='y', alpha=0.3)
plt.show()

# ==========================================
# 3. SAVE TO CSVs
# ==========================================
df_median.to_csv(OUTPUT_CSV_MEDIAN, index=False)
df_std.to_csv(OUTPUT_CSV_STD, index=False)

print(f"Median data saved to: {OUTPUT_CSV_MEDIAN}")
print(f"Std Dev data saved to: {OUTPUT_CSV_STD}")

In [ ]:
#count patches

In [ ]:
import numpy as np
import os
from tqdm import tqdm

PROJECT_ROOT = r'E:\Thesis3april'
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
          "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
          "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

total_patches = 0
total_files = 0

print("Calculating total patch count...")

for plate in PLATES:
    feature_path = os.path.join(PROJECT_ROOT, "features", plate)
    
    if not os.path.exists(feature_path):
        print(f"Skipping {plate}: Path not found.")
        continue
        
    # Walk through all well subfolders
    for root, dirs, files in os.walk(feature_path):
        for file in files:
            if file.endswith(".npz"):
                file_path = os.path.join(root, file)
                try:
                    with np.load(file_path) as data:
                        # 'features' is the standard key in DeepProfiler npz files
                        # We only need the shape[0] (number of rows/cells)
                        total_patches += data["features"].shape[0]
                        total_files += 1
                except Exception as e:
                    print(f"Could not read {file}: {e}")

print("\n--- Final Statistics ---")
print(f"Total .npz files (sites) processed: {total_files}")
print(f"Total number of patches (cells):     {total_patches:,}")

In [ ]:
#remove mutants less than 5 patches

In [ ]:
import pandas as pd
import os

# 1. SETUP
PROJECT_ROOT = r'E:\Thesis3april'
INPUT_CSV = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "files")

# Create the directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. DEFINE YOUR NEW THRESHOLD
STRICT_THRESHOLD = 5 

# 3. LOAD & FILTER
print(f"Loading {INPUT_CSV}...")
df = pd.read_csv(INPUT_CSV)

# Identify the wells that ARE BELOW the threshold
removed_df = df[df['Cell_Count'] < STRICT_THRESHOLD].copy()

# Identify the wells that ARE ABOVE or EQUAL to the threshold
filtered_df = df[df['Cell_Count'] >= STRICT_THRESHOLD].copy()

# 4. REPORT & SAVE
print(f"\n--- Filtering Summary ---")
print(f"Original wells:       {len(df)}")
print(f"Wells kept:           {len(filtered_df)}")
print(f"Wells removed:        {len(removed_df)}")
print(f"-------------------------")

if not removed_df.empty:
    print(f"\n--- LIST OF REMOVED WELLS (Count < {STRICT_THRESHOLD}) ---")
    # We only show the metadata columns for the removed wells
    print(removed_df[['Plate', 'Well_ID', 'Treatment', 'Cell_Count']].to_string(index=False))
else:
    print("\nNo wells were below the threshold.")

# 5. SAVE DATA
OUTPUT_CSV_FILTERED = os.path.join(OUTPUT_DIR, f"aggregated_wells_median_min5.csv")
filtered_df.to_csv(OUTPUT_CSV_FILTERED, index=False)

print(f"\nDone! Filtered data saved to: {OUTPUT_CSV_FILTERED}")

In [ ]:
#plot without feature selection

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 0. ADAPTABLE STYLE PARAMETERS ---
# Axis Style
AXIS_LINE_WIDTH = 3       # Thickness of the L-shaped axis lines
AXIS_TICK_WIDTH = 3       # Thickness of the tick marks
AXIS_TICK_LEN = 8         # Length of the tick marks
AXIS_TICK_FONT_SIZE = 25  # Size of the numbers (0, 5, 10...) on the axis
AXIS_TITLE_FONT_SIZE = 25 # Size of "UMAP 1" and "UMAP 2" text

# Legend Style
LEGEND_FONT_SIZE = 25     # Font size for the legend text
LEGEND_SYMBOL_SIZE = 14   # Size of the symbols in the legend list
HEADER_SYMBOL_SIZE = 14   # Size of the Square/Circle symbols in the header

# Plot Marker Style
MARKER_SIZE = 8           # Size of the dots/squares in the actual plot
MARKER_OPACITY = 0.8      # Transparency of the points (0 to 1)

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "All_Plates_NofeatureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

# Create directories
for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)


df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. SELECTION & MAPPING ---
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no-xyl", "T1": "xyl5", "T2": "xyl12"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

custom_colors = [
    '#5DADE2', '#D988B9', '#52BE80', '#E74C3C', '#AF7AC5', 
    '#707B7C', '#B7950B', '#8E44AD', '#E59896', '#5D3FD3', 
    '#45B39D', '#F333FF', '#FF3385', '#C39BD3', '#7D6608'
]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing: {config['name']} ({len(valid_indices)} features)...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Save coordinates
    meta_cols = ['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']
    available_meta = [c for c in meta_cols if c in df.columns]
    df_coords = df[available_meta].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_coords.to_csv(os.path.join(COORD_DIR, f"Coords_{config['name'].replace(' ', '_')}.csv"), index=False)

    # --- CREATE PLOT ---
    fig = go.Figure()

    # Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no-sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # Data Traces
    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        color = custom_colors[i % len(custom_colors)]
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)
                ),
                text=sub_data['Treatment'], hoverinfo='text+name'
            ))

    # Layout
    fig.update_layout(
        width=900, height=900, template='plotly_white',
        font=dict(family='Arial'),
        
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE), 
          
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE) 
            
        ),
        
    )

    # Save
    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

In [ ]:
#time overlay on everything without feature selection

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3
AXIS_TICK_WIDTH = 3
AXIS_TICK_LEN = 8
AXIS_TICK_FONT_SIZE = 24
AXIS_TITLE_FONT_SIZE = 24

LEGEND_FONT_SIZE = 24
LEGEND_SYMBOL_SIZE = 14
HEADER_SYMBOL_SIZE = 14

MARKER_SIZE = 8
MARKER_OPACITY = 0.8

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "AllPlates_noSelection_Time_Overlay")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for folder in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. PREPROCESSING & MAPPING ---
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
time_display_map = {'T0': 'no-xyl', 'T1': 'xyl5', 'T2': 'xyl12'}
df['Time_Display'] = df['Timepoint'].map(time_display_map)
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# Updated Color Mapping with Mutant/Control differences
time_colors = {
    'no-xyl': {'Mutant': '#FFA07A', 'Control': "#B13E17"}, # Light Salmon vs Darker Coral
    'xyl5':   {'Mutant': '#4DB6AC', 'Control': "#036257"}, # Light Teal vs Deep Teal
    'xyl12':  {'Mutant': '#F0D05D', 'Control': '#B8860B'}  # Light Goldenrod vs Dark Goldenrod
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Generating Time Overlay for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df[['Plate', 'Well_ID', 'Treatment', 'Type', 'Time_Display']].copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    coord_file = f"Coords_Time_{config['name'].replace(' ', '_')}.csv"
    df_plot.to_csv(os.path.join(COORD_DIR, coord_file), index=False)

    fig = go.Figure()

    # A) Add Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no-sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # B) Add Data Traces grouped by Time
    times = ['no-xyl', 'xyl5', 'xyl12']
    for t_val in times:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Time_Display'] == t_val) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            # Access nested color based on Type
            color = time_colors[t_val][t_type]
            
            fig.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=t_val,
                legendgroup=t_val,
                showlegend=True if t_type == 'Mutant' else False,
                marker=dict(
                    color=color, 
                    size=MARKER_SIZE,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=MARKER_OPACITY
                ),
                text=curr['Treatment'],
                hoverinfo='text+name'
            ))

    # C) Final Layout
    fig.update_layout(
        width=900, height=900,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    base_name = f"UMAP_Time_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)

print("\nDone. Time overlay plots and coordinates are saved.")

In [ ]:
#feature seleciton on everything 

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
INPUT_CSV = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "files")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# Helper function to display control counts
def print_control_summary(df, step_name):
    controls = df[df['Treatment'] == CONTROL_LABEL]
    count = len(controls)
    print(f"[{step_name}] Controls remaining: {count}")
    # Optional: Breakdown by plate
    # print(controls.groupby('Plate').size())

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

print(f"Total starting features: {len(feature_cols)}")
print_control_summary(df_raw, "Initial")

# --- STEP 0: GLOBAL VARIANCE FILTER ---
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
removed_0 = len(feature_cols) - len(active_features)
print(f"-> Removed: {removed_0} | Remaining: {len(active_features)}")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"\nRunning Step 1: Within-Plate Consistency (Top 2000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
removed_1 = len(active_features) - len(step1_features)
print(f"-> Removed: {removed_1} | Remaining: {len(step1_features)}")
print_control_summary(df_raw, "Step 1")

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"\nRunning Step 2: Across-Plate Stability (Top 200)...")
step2_features = filter_across_plate_stability(df_raw, step1_features, top_n_to_keep=200)
removed_2 = len(step1_features) - len(step2_features)
print(f"-> Removed: {removed_2} | Remaining: {len(step2_features)}")
print_control_summary(df_raw, "Step 2")

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"\nRunning Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_raw, step2_features, correlation_threshold=0.9)
removed_3 = len(step2_features) - len(final_feature_list)
print(f"-> Removed: {removed_3} | Remaining: {len(final_feature_list)}")
print_control_summary(df_raw, "Step 3")

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_raw[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Total features filtered out: {len(feature_cols) - len(final_feature_list)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#aantal controles

In [ ]:
# Assuming 'df_final' is your final dataframe after all filtering
# and 'CONTROL_LABEL' is "no_sgRNA"

def summarize_final_controls(df, control_label):
    # Filter for controls only
    controls_df = df[df['Treatment'] == control_label].copy()
    
    # Create a Timepoint column based on the Plate name
    # This looks for T0, T1, or T2 within the Plate string
    controls_df['Timepoint'] = controls_df['Plate'].str.extract(r'(T[0-2])')
    
    # Count occurrences
    counts = controls_df.groupby('Timepoint').size()
    
    print("--- Final Control Counts per Condition ---")
    if counts.empty:
        print("No controls found. Check if CONTROL_LABEL matches your data.")
    else:
        print(counts)
    
    return counts

# Run it
summarize_final_controls(df_final, CONTROL_LABEL)

In [ ]:
#Plotting all after featue seleciton

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 0. ADAPTABLE STYLE PARAMETERS ---
# Axis Style
AXIS_LINE_WIDTH = 3       # Thickness of the L-shaped axis lines
AXIS_TICK_WIDTH = 3       # Thickness of the tick marks
AXIS_TICK_LEN = 8         # Length of the tick marks
AXIS_TICK_FONT_SIZE = 24  # Size of the numbers (0, 5, 10...) on the axis
AXIS_TITLE_FONT_SIZE = 24 # Size of "UMAP 1" and "UMAP 2" text

# Legend Style
LEGEND_FONT_SIZE = 24     # Font size for the legend text
LEGEND_SYMBOL_SIZE = 14   # Size of the symbols in the legend list
HEADER_SYMBOL_SIZE = 14   # Size of the Square/Circle symbols in the header

# Plot Marker Style
MARKER_SIZE = 8           # Size of the dots/squares in the actual plot
MARKER_OPACITY = 0.8      # Transparency of the points (0 to 1)

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "All_Plates_featureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. SELECTION & MAPPING ---
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no-xyl", "T1": "xyl5", "T2": "xyl12"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

custom_colors = [
    '#5DADE2', '#D988B9', '#52BE80', '#E74C3C', '#AF7AC5', 
    '#707B7C', '#B7950B', '#8E44AD', '#E59896', '#5D3FD3', 
    '#45B39D', '#F333FF', '#FF3385', '#C39BD3', '#7D6608'
]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing: {config['name']} ({len(valid_indices)} features)...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Save coordinates
    meta_cols = ['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']
    available_meta = [c for c in meta_cols if c in df.columns]
    df_coords = df[available_meta].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_coords.to_csv(os.path.join(COORD_DIR, f"Coords_{config['name'].replace(' ', '_')}.csv"), index=False)

    # --- CREATE PLOT ---
    fig = go.Figure()

    # Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no-sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # Data Traces
    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        color = custom_colors[i % len(custom_colors)]
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)
                ),
                text=sub_data['Treatment'], hoverinfo='text+name'
            ))

    # Layout
    fig.update_layout(
        width=900, height=900, template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    # Save
    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=2)
    
    print(f"Finished {config['name']}")

In [ ]:
#all met iets lichtere gedient maar mschn minder goed

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.colors as mc
import colorsys

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      

# Helper function to lighten/darken colors
def adjust_color(color, amount=0.5):
    """
    Lightens the given color by multiplying (1-luminance) by the amount.
    Input can be matplotlib color string, hex string, or RGB tuple.
    amount > 1.0 = lighter; amount < 1.0 = darker.
    """
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return mc.to_hex(colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2]))

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots", "All_Plates_featureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. SELECTION & MAPPING ---
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl12"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

# Your base colors
custom_colors = [
    '#5DADE2', '#D988B9', '#52BE80', '#E74C3C', '#AF7AC5', 
    '#707B7C', '#B7950B', '#8E44AD', '#E59896', '#5D3FD3', 
    '#45B39D', '#F333FF', '#FF3385', '#C39BD3', '#7D6608'
]

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Processing: {config['name']} ({len(valid_indices)} features)...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_coords = df[['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]
    df_coords.to_csv(os.path.join(COORD_DIR, f"Coords_{config['name'].replace(' ', '_')}.csv"), index=False)

    fig = go.Figure()

    # Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # Data Traces
    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        base_color = custom_colors[i % len(custom_colors)]
        
        # Mutant = Lighter, Control = Darker
        mutant_color = adjust_color(base_color, amount=1.2) 
        control_color = adjust_color(base_color, amount=0.7) 
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            
            color = mutant_color if t_type == 'Mutant' else control_color
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)
                ),
                text=sub_data['Treatment'], hoverinfo='text+name'
            ))

    fig.update_layout(
        width=900, height=900, template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(itemsizing='constant', font=dict(size=LEGEND_FONT_SIZE)),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
           
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            
        )
    )

    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    # Updated to scale=7 for 500 DPI requirement
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

In [ ]:
#time overlay

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "All_Plates_FeatureSelection_Time_Overlay")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

# --- 2. PREPROCESSING & MAPPING ---
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
time_display_map = {'T0': 'no-xyl', 'T1': 'xyl5', 'T2': 'xyl12'}
df['Time_Display'] = df['Timepoint'].map(time_display_map)

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# Color Mapping: Sub-divided into Mutant (Lighter) and Control (Darker)
# Base Coral: #FF7F50 | Base Teal: #008080 | Base Goldenrod: #DAA520
time_colors = {
    'no-xyl': {'Mutant': '#FFA07A', 'Control': "#B13E17"}, # Light Salmon vs Darker Coral
    'xyl5':   {'Mutant': '#4DB6AC', 'Control': "#036257"}, # Light Teal vs Deep Teal
    'xyl12':  {'Mutant': '#F0D05D', 'Control': '#B8860B'}  # Light Goldenrod vs Dark Goldenrod
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    print(f"Generating Time Overlay for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df[['Plate', 'Well_ID', 'Treatment', 'Type', 'Time_Display']].copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    coord_file = f"Coords_Time_{config['name'].replace(' ', '_')}.csv"
    df_plot.to_csv(os.path.join(COORD_DIR, coord_file), index=False)

    fig = go.Figure()

    # A) Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no-sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # B) Data Traces
    times = ['no-xyl', 'xyl5', 'xyl12']
    for t_val in times:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Time_Display'] == t_val) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            # Select the specific shade for this type
            color = time_colors[t_val][t_type]
            
            fig.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=t_val,
                legendgroup=t_val,
                showlegend=True if t_type == 'Mutant' else False,
                marker=dict(
                    color=color, 
                    size=MARKER_SIZE,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    # Controls have a black outline to make them pop even more
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=MARKER_OPACITY
                ),
                text=curr['Treatment'],
                hovertemplate="<b>%{text}</b><br>Condition: %{name}<extra></extra>"
            ))

    # C) Final Layout
    fig.update_layout(
        width=900, height=900,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE)
        )
    )

    base_name = f"UMAP_Time_{config['name'].replace(' ', '_')}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    # scale=7 ensures ~500 DPI for a 900x900 base image
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)

print("\nDone. 500 DPI PNGs, SVGs, and coordinates are saved.")

In [ ]:

# 3.5 FEATURE SUMMARY PRINT-OUT
print("\n" + "="*45)
print(f"{'Channel Selection':<25} | {'Features Found':<15}")
print("-" * 45)
for config in plot_configs:
    print(f"{config['name']:<25} | {len(config['indices']):<15}")
print("="*45 + "\n")

In [ ]:
#enkelT0enT1 bekijken
###
#

In [ ]:
#no xylose and xylose 5h plates without feature selection

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.colors as mc
import colorsys

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3        
AXIS_TICK_WIDTH = 3        
AXIS_TICK_LEN = 8          
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8            
MARKER_OPACITY = 0.8      

# Helper function to lighten/darken colors
def adjust_color(color, amount=0.5):
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return mc.to_hex(colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2]))

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv")
# Updated folder name based on your second snippet
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_No_featureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1","PLATE3_T0","PLATE3_T1",
                   "PLATE4_T0","PLATE4_T1","PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl10"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

custom_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                 '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_coords = df[['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- SAVE COORDINATES ---
    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    df_coords.to_csv(os.path.join(COORD_DIR, f"{base_name}_coords.csv"), index=False)

    fig = go.Figure()

    # --- UPDATED LEGEND HEADERS ---
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no-sgRNA', showlegend=True))
    
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True))

    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        base_color = custom_colors[i % len(custom_colors)]
        mutant_color = adjust_color(base_color, amount=1.2) 
        control_color = adjust_color(base_color, amount=0.7) 
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            color = mutant_color if t_type == 'Mutant' else control_color
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(symbol='circle' if t_type == 'Mutant' else 'square',
                            size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                            line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)),
                text=sub_data['Treatment'], hoverinfo='text+name'))

    # --- UPDATED LAYOUT FOR CONSISTENT SQUARE PLOT AREA ---
    # Layout
    fig.update_layout(
        width=1200, height=900, template='plotly_white',                      # met de scalanchor en scale ratio forceer je x en y as tick scale gelijk, maar plot is breeder dan hoog, als je dus square forceert (900 x 900) dan is er veel witruimte
        font=dict(family='Arial'),
        
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE), 
            scaleanchor="x", # Forces square aspect ratio
            scaleratio=1,
          
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            scaleanchor="x", # Forces square aspect ratio
            scaleratio=1,
            
        ),
        
    )

    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

print("\nDone. Plots and coordinates generated successfully.")

In [ ]:
#time ovarlay no xylose and xylose 5h without feature selection

In [ ]:
import pandas as pd
import os
import plotly.graph_objects as go

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_No_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_No_featureSelection_Time_Overlay")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Color Mapping for Timepoints
time_colors = {
    'no_xyl': {'Mutant': '#FFA07A', 'Control':  "#B13E17"}, 
    'xyl5':   {'Mutant': '#4DB6AC', 'Control': "#036257"}, 
}


# List the coordinate files you want to process
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. LOOP THROUGH EXISTING COORDINATES ---
for file_name in coord_files:
    print(f"Plotting Time Overlay from: {file_name}...")
    
    # Load existing coordinates
    df_plot = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Ensure Time_Display is available (mapping if needed)
    if 'Time_Display' not in df_plot.columns:
        df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'(T\d+)')
        time_display_map = {'T0': 'no_xyl', 'T1': 'xyl5', 'T2': 'xyl10'}
        df_plot['Time_Display'] = df_plot['Timepoint'].map(time_display_map)

    fig = go.Figure()

    # A) Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # B) Data Traces
    times = ['no_xyl', 'xyl5', 'xyl10']
    for t_val in times:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Time_Display'] == t_val) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            color = time_colors.get(t_val, {}).get(t_type, '#CCCCCC') # Default gray if time not in map
            
            fig.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=t_val,
                legendgroup=t_val,
                showlegend=True if t_type == 'Mutant' else False,
                marker=dict(
                    color=color, 
                    size=MARKER_SIZE,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=MARKER_OPACITY
                ),
                text=curr['Treatment'],
                hovertemplate="<b>%{text}</b><br>Condition: %{name}<extra></extra>"
            ))

    # C) Final Layout (Consistent Square Plot Area)
    fig.update_layout(
        width=1200, height=900, template='plotly_white',                      # met de scalanchor en scale ratio forceer je x en y as tick scale gelijk, maar plot is breeder dan hoog, als je dus square forceert (900 x 900) dan is er veel witruimte
        font=dict(family='Arial'),
        
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE)
        ),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE), 
            scaleanchor="x", # Forces square aspect ratio
            scaleratio=1,
          
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            scaleanchor="x", # Forces square aspect ratio
            scaleratio=1,
            
        ),
        
    )

    base_name = file_name.replace("Coords_", "UMAP_Time_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=7)
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Time overlay plots generated with fixed square axes.")

In [ ]:
#preselection T0 en T1 enkel 

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
INPUT_CSV = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "files")
CONTROL_LABEL = "no_sgRNA" 

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Loading raw data...")
df_raw_full = pd.read_csv(INPUT_CSV)
df_raw_full.columns = [str(c) for c in df_raw_full.columns]

# --- NEW STEP: FILTER OUT T2 AND SAVE COPY ---
print("Filtering out T2 samples...")
df_raw = df_raw_full[df_raw_full['Plate'].str.contains('T0|T1')].copy()

filtered_raw_path = os.path.join(OUTPUT_DIR, "raw_data_T0_T1_only.csv")
df_raw.to_csv(filtered_raw_path, index=False)

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

print(f"\nInitial total features: {len(feature_cols)}")

# --- STEP 0: GLOBAL VARIANCE FILTER ---
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
removed_0 = len(feature_cols) - len(active_features)
print(f"-> Removed: {removed_0} | Remaining: {len(active_features)}")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"\nRunning Step 1: Within-Plate Consistency (Top 2000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
removed_1 = len(active_features) - len(step1_features)
print(f"-> Removed: {removed_1} | Remaining: {len(step1_features)}")

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"\nRunning Step 2: Across-Plate Stability (Top 200)...")
step2_features = filter_across_plate_stability(df_raw, step1_features, top_n_to_keep=200)
removed_2 = len(step1_features) - len(step2_features)
print(f"-> Removed: {removed_2} | Remaining: {len(step2_features)}")

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"\nRunning Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_raw, step2_features, correlation_threshold=0.9)
removed_3 = len(step2_features) - len(final_feature_list)
print(f"-> Removed: {removed_3} | Remaining: {len(final_feature_list)}")

# ==========================================
# FINAL SAVE & SUMMARY
# ==========================================
df_final = df_raw[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_T0_T1.csv")
df_final.to_csv(output_path, index=False)

# Get control counts per timepoint for the final file
final_controls = df_final[df_final['Treatment'] == CONTROL_LABEL].copy()
final_controls['Timepoint'] = final_controls['Plate'].str.extract(r'(T[0-1])')
control_counts = final_controls.groupby('Timepoint').size()

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE (T0 & T1 ONLY)")
print(f"Total features removed: {len(feature_cols) - len(final_feature_list)}")
print(f"Final feature count: {len(final_feature_list)}")
print("\n--- Final Control Counts per Condition ---")
print(control_counts if not control_counts.empty else "No controls found.")
print("="*40)

In [ ]:
#plot enkel T0 en T1

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.colors as mc
import colorsys

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3        
AXIS_TICK_WIDTH = 3        
AXIS_TICK_LEN = 8          
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8            
MARKER_OPACITY = 0.8      

# Helper function to lighten/darken colors
def adjust_color(color, amount=0.5):
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return mc.to_hex(colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2]))

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_T0_T1.csv")
# Updated folder name based on your second snippet
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1","PLATE3_T0","PLATE3_T1",
                   "PLATE4_T0","PLATE4_T1","PLATE5_T0","PLATE5_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl10"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

custom_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
                 '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_coords = df[['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- SAVE COORDINATES ---
    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    df_coords.to_csv(os.path.join(COORD_DIR, f"{base_name}_coords.csv"), index=False)

    fig = go.Figure()

    # --- UPDATED LEGEND HEADERS ---
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no-sgRNA', showlegend=True))
    
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True))

    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        base_color = custom_colors[i % len(custom_colors)]
        mutant_color = adjust_color(base_color, amount=1.2) 
        control_color = adjust_color(base_color, amount=0.7) 
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            color = mutant_color if t_type == 'Mutant' else control_color
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(symbol='circle' if t_type == 'Mutant' else 'square',
                            size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                            line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)),
                text=sub_data['Treatment'], hoverinfo='text+name'))

    # --- UPDATED LAYOUT FOR CONSISTENT SQUARE PLOT AREA ---
    fig.update_layout(
        width=1400,  
        height=900,  
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               
            y=1, 
            xanchor='left',
            yanchor='top',
        ),
        
        margin=dict(l=100, r=350, t=80, b=100),

        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] 
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", 
            scaleratio=1,
            constrain='domain'
        )
    )

    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

print("\nDone. Plots and coordinates generated successfully.")

In [ ]:

# 3.5 FEATURE SUMMARY PRINT-OUT
print("\n" + "="*45)
print(f"{'Channel Selection':<25} | {'Features Found':<15}")
print("-" * 45)
for config in plot_configs:
    print(f"{config['name']:<25} | {len(config['indices']):<15}")
print("="*45 + "\n")

In [ ]:
#time overlay T0 en T1

In [ ]:
import pandas as pd
import os
import plotly.graph_objects as go

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_Time_Overlay")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


# Color Mapping for Timepoints
time_colors = {
    'no_xyl': {'Mutant': '#FFA07A', 'Control':  "#B13E17"}, 
    'xyl5':   {'Mutant': '#4DB6AC', 'Control': "#036257"}, 
}
# List the coordinate files you want to process
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. LOOP THROUGH EXISTING COORDINATES ---
for file_name in coord_files:
    print(f"Plotting Time Overlay from: {file_name}...")
    
    # Load existing coordinates
    df_plot = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Ensure Time_Display is available (mapping if needed)
    if 'Time_Display' not in df_plot.columns:
        df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'(T\d+)')
        time_display_map = {'T0': 'no_xyl', 'T1': 'xyl5', 'T2': 'xyl10'}
        df_plot['Time_Display'] = df_plot['Timepoint'].map(time_display_map)

    fig = go.Figure()

    # A) Legend Header
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no_sgRNA', showlegend=True
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True
    ))

    # B) Data Traces
    times = ['no_xyl', 'xyl5', 'xyl10']
    for t_val in times:
        for t_type in ['Mutant', 'Control']:
            mask = (df_plot['Time_Display'] == t_val) & (df_plot['Type'] == t_type)
            curr = df_plot[mask]
            if curr.empty: continue
            
            color = time_colors.get(t_val, {}).get(t_type, '#CCCCCC') # Default gray if time not in map
            
            fig.add_trace(go.Scatter(
                x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                name=t_val,
                legendgroup=t_val,
                showlegend=True if t_type == 'Mutant' else False,
                marker=dict(
                    color=color, 
                    size=MARKER_SIZE,
                    symbol='circle' if t_type == 'Mutant' else 'square',
                    line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0),
                    opacity=MARKER_OPACITY
                ),
                text=curr['Treatment'],
                hovertemplate="<b>%{text}</b><br>Condition: %{name}<extra></extra>"
            ))

    # C) Final Layout (Consistent Square Plot Area)
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               # Positioned in the space created by the domain
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), # Increased right margin for legend
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, # L-shaped
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] # Reserves 25% of the width for legend/margin
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, # L-shaped
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", # Forces square aspect ratio
            scaleratio=1,
            constrain='domain'
        )
    )

    base_name = file_name.replace("Coords_", "UMAP_Time_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=7)
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Time overlay plots generated with fixed square axes.")

In [ ]:
#time overlay with contour

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 4       
AXIS_TICK_WIDTH = 4       
AXIS_TICK_LEN = 10          
AXIS_TICK_FONT_SIZE = 30  
AXIS_TITLE_FONT_SIZE = 30 

LEGEND_FONT_SIZE = 35      
HEADER_SYMBOL_SIZE = 35    

MARKER_SIZE = 8            
MARKER_OPACITY = 0.8       
PNG_RESOLUTION_SCALE = 7

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_Time_Contour_Overlay")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Color Mapping for Timepoints (Mutants only)
time_colors = {
    'no_xyl': '#FFA07A', 
    'xyl5':   '#4DB6AC', 
   
}

coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Time Overlay + Contours for: {file_name}...")
    
    df_plot = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Ensure Time_Display logic
    if 'Time_Display' not in df_plot.columns:
        df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'(T\d+)')
        time_display_map = {'T0': 'no_xyl', 'T1': 'xyl5', 'T2': 'xyl10'}
        df_plot['Time_Display'] = df_plot['Timepoint'].map(time_display_map)

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    # Calculate density from Control points (even if we don't plot the points themselves)
    ctrl_df = df_plot[df_plot['Type'] == 'Control']
    
    if not ctrl_df.empty and len(ctrl_df) > 1:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        
        # Calculate thresholds for density levels (95, 70, 45, 20% density)
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        # Create grid for contours
        x_range = df_plot['UMAP1'].max() - df_plot['UMAP1'].min()
        y_range = df_plot['UMAP2'].max() - df_plot['UMAP2'].min()
        
        x_grid = np.linspace(df_plot['UMAP1'].min() - x_range*0.1, df_plot['UMAP1'].max() + x_range*0.1, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min() - y_range*0.1, df_plot['UMAP2'].max() + y_range*0.1, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        # Extract paths using matplotlib and add to Plotly
        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.5),
                    showlegend=False, hoverinfo='skip'
                ))
        plt.close(fig_tmp)

    # --- 4. LEGEND HEADERS ---
    # Visual indicator for what the grey lines represent
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        line=dict(color='#808080', width=2),
        name='no_sgRNA Density', showlegend=True
    ))
    # Header for the colored circles
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutants', showlegend=True
    ))

    # --- 5. DATA TRACES (Mutants Only) ---
    times = ['no_xyl', 'xyl5', 'xyl10']
    for t_val in times:
        mask = (df_plot['Time_Display'] == t_val) & (df_plot['Type'] == 'Mutant')
        curr = df_plot[mask]
        if curr.empty: continue
        
        color = time_colors.get(t_val, '#CCCCCC')
        
        fig.add_trace(go.Scatter(
            x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
            name=t_val,
            legendgroup=t_val,
            showlegend=True,
            marker=dict(
                color=color, 
                size=MARKER_SIZE,
                symbol='circle',
                line=dict(width=0.4, color='white'),
                opacity=MARKER_OPACITY
            ),
            text=curr['Treatment'],
            hovertemplate="<b>%{text}</b><br>Time: " + t_val + "<extra></extra>"
        ))

    # --- 6. LAYOUT (Fixed Square Plot Area) ---
    fig.update_layout(
        width=1400, height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82, y=1, 
            xanchor='left', yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] 
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", 
            scaleratio=1,
            constrain='domain'
        )
    )

    # --- 7. SAVE ---
    base_name = file_name.replace("Coords_", "UMAP_Time_Contour_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Plots saved with control contours and mutant-only points.")

In [ ]:
#annotation T0T1, met nieuwe kleur

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 16     
LEGEND_TITLE_SIZE = 16    
LEGEND_SYMBOL_SIZE = 14   

MARKER_SIZE_MUTANT = 8    
MARKER_SIZE_CONTROL = 10  
MARKER_OPACITY = 0.8      

PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
# We load the annotation file as requested
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

# Points to coordinates generated in your first run (e.g., Coords_Channel_1.csv)
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_Annotated_Pathways")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# EXPLICIT COLOR MAP
MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                      
    "biosynthesis of fatty acids": "#9467bd",    
    "DNA replication": "#8c564b",                
    "DNA condensation/ segregation": "#e377c2",  
    "biosynthesis of isoprenoids": "#7f7f7f",    
    "cell division": "#bcbd22",                  
    "ribosomal proteins": "#17becf",             
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                      
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#333333",  
    "Baseline (T0)": "#808080",  
    "Unknown/Other": "#D3D3D3"   
}

# --- 2. LOAD ANNOTATIONS ---
anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 3. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Annotating and Plotting: {file_name}...")
    
    # Load pre-calculated UMAP coordinates
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Merge with Annotations based on Treatment
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], 
                              on='Treatment', how='left')

    # Re-apply the Annotation cleaning logic
    df_plot['SubtiWiki Annotation 4'] = df_plot['SubtiWiki Annotation 4'].astype(str).str.strip()
    df_plot['SubtiWiki Annotation 3'] = df_plot['SubtiWiki Annotation 3'].astype(str).str.strip()
    
    df_plot['Effective_Annotation'] = (df_plot['SubtiWiki Annotation 4']
                                       .replace('nan', np.nan)
                                       .fillna(df_plot['SubtiWiki Annotation 3'].replace('nan', np.nan))
                                       .fillna("Unknown/Other"))
    
    # Determine Timepoint if not present
    if 'Timepoint' not in df_plot.columns:
        df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')

    # Category Logic
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    df_plot['Display_Category'] = df_plot['Effective_Annotation']
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend Order
    unique_in_data = df_plot['Display_Category'].unique()
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["Unknown/Other"]

    auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
    color_lookup = MANUAL_COLORS.copy()
    for i, cat in enumerate(other_cats):
        color_lookup[cat] = auto_pal[i % len(auto_pal)]

    fig = go.Figure()

    # Draw traces in reverse for layering
    for cat in final_legend_order[::-1]:
        if cat not in df_plot['Display_Category'].values: continue
        cat_df = df_plot[df_plot['Display_Category'] == cat]
        is_c_cat = (cat == "Control Group")

        sub_groups = [
            ('circle', cat_df[~cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])]),
            ('square', cat_df[(cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])) & (cat_df['Timepoint'] == 'T0')]),
            ('diamond', cat_df[(cat_df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])) & (cat_df['Timepoint'] == 'T1')])
        ]

        for sym, sub_df in sub_groups:
            if sub_df.empty: continue
            sz = MARKER_SIZE_CONTROL if is_c_cat else MARKER_SIZE_MUTANT
            clr = color_lookup.get(cat, "#000000")

            fig.add_trace(go.Scatter(
                x=sub_df['UMAP1'], y=sub_df['UMAP2'],
                mode='markers', name=cat, legendgroup=cat, showlegend=False,
                marker=dict(
                    size=sz, color=clr, symbol=sym,
                    opacity=MARKER_OPACITY + 0.1 if is_c_cat else MARKER_OPACITY,
                    line=dict(width=0.4, color='white')
                ),
                text=sub_df['Treatment'],
                hovertemplate="<b>%{text}</b><br>Category: " + cat + "<extra></extra>"
            ))

    # Manage Legend Visibility
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            if trace.name == cat and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    # --- THE SYNCHRONIZED SQUARE LAYOUT ---
    # C) Final Layout (Consistent Square Plot Area)
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               # Positioned in the space created by the domain
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), # Increased right margin for legend
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, # L-shaped
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] # Reserves 25% of the width for legend/margin
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, # L-shaped
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", # Forces square aspect ratio
            scaleratio=1,
            constrain='domain'
        )
    )

    base_name = file_name.replace("Coords_", "UMAP_Annotated_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Fast annotated plots generated using pre-calculated coordinates.")

In [ ]:
#mooiste contour

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

# Using existing coordinates folder
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_Contour")

for d in [OUTPUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "unknown function": "#000000",
   
}

# Load Annotations
anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Contour Overlay for: {file_name}...")
    
    # Load coordinates
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Re-merge and clean metadata for category logic
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    # Category setup
    df_plot['Display_Category'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3']).fillna("unknown function")
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend Logic
    unique_in_data = df_plot['Display_Category'].unique()
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["unknown function"]
    
    color_lookup = MANUAL_COLORS.copy()
    auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
    for i, cat in enumerate(other_cats):
        color_lookup[cat] = auto_pal[i % len(auto_pal)]

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        
        # Calculate thresholds for 95, 70, 45, 20% density
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        # Create grid for contours
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        # Use matplotlib to extract contour paths
        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.5),
                    showlegend=False, hoverinfo='skip',
                    name='Control Group' if i == 0 and j == 0 else 'inner'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS ---
    for cat in final_legend_order[::-1]:
        if cat not in df_plot['Display_Category'].values or cat == "Control Group": continue
        sub_df = df_plot[df_plot['Display_Category'] == cat]
        
        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], y=sub_df['UMAP2'], mode='markers',
            name=cat, legendgroup=cat, showlegend=False,
            marker=dict(size=MARKER_SIZE, color=color_lookup.get(cat, "#000000"),
                        opacity=MARKER_OPACITY, line=dict(width=0.4, color='white')),
            text=sub_df['Treatment'],
            hovertemplate="<b>%{text}</b><br>Cat: " + cat + "<extra></extra>"
        ))

    # --- 5. FIXED SQUARE LAYOUT (70/30 SPLIT) ---
     # C) Final Layout (Consistent Square Plot Area)
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               # Positioned in the space created by the domain
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), # Increased right margin for legend
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, # L-shaped
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] # Reserves 25% of the width for legend/margin
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, # L-shaped
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", # Forces square aspect ratio
            scaleratio=1,
            constrain='domain'
        )
    )

    # Final legend visibility override
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            if (trace.name == cat or (cat == 'Control Group' and trace.name == 'Control Group')) and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    base_name = file_name.replace("Coords_", "UMAP_Contour_")
    
    # Saving outputs (PNG, SVG, and HTML)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Contour plots generated (PNG, SVG, HTML) with consistent 70/30 square logic.")

In [ ]:
#14 april mooiste contour, andere kleuren

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "14april_T0T1_Contour")

for d in [OUTPUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9040db",  
    "ribosomal proteins": "#17becf",    
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "cell division": "#bcbd22",                    
                 
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "unknown function": "#292727",
}

# Load Annotations
anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Contour Overlay for: {file_name}...")
    
    # Load coordinates
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Re-merge and clean metadata
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    # Category setup
    df_plot['Display_Category'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3']).fillna("unknown function")
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend and Color Logic
    unique_in_data = df_plot['Display_Category'].unique()
    
    # Identify categories not in MANUAL_COLORS to put them in the legend and color them black
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:12] + other_cats + ["unknown function"]
    
    # Create the lookup: if not in MANUAL_COLORS, default to black
    color_lookup = MANUAL_COLORS.copy()
    for cat in other_cats:
        color_lookup[cat] = "#292727"

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.5),
                    showlegend=False, hoverinfo='skip',
                    name='Control Group' if i == 0 and j == 0 else 'inner'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS ---
    for cat in final_legend_order[::-1]:
        # Controls are not plotted as points, only contours
        if cat not in df_plot['Display_Category'].values or cat == "Control Group": 
            continue
            
        sub_df = df_plot[df_plot['Display_Category'] == cat]
        
        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], y=sub_df['UMAP2'], mode='markers',
            name=cat, legendgroup=cat, showlegend=False,
            marker=dict(size=MARKER_SIZE, 
                        color=color_lookup.get(cat, "#000000"),
                        opacity=MARKER_OPACITY, 
                        line=dict(width=0.4, color='white')),
            text=sub_df['Treatment'],
            hovertemplate="<b>%{text}</b><br>Cat: " + cat + "<extra></extra>"
        ))

    # --- 5. FIXED SQUARE LAYOUT ---
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82, 
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), 
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False,
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] 
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False,
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", 
            scaleratio=1,
            constrain='domain'
        )
    )

    # Final legend visibility override
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            # Show legend for points OR for the control category (even if points aren't plotted)
            if (trace.name == cat or (cat == 'Control Group' and trace.name == 'Control Group')) and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                # Ensure ranking matches desired order
                try:
                    trace.legendrank = final_legend_order.index(cat)
                except ValueError:
                    pass

    base_name = file_name.replace("Coords_", "UMAP_Contour_")
    
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Datapoints not in manual list are now black; T0 remains grey; controls are contour-only.")

In [ ]:
#contour annotaitons enkel denege uit lijst gekleurd

In [ ]:
#perpathway apart

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8          
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 8           
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
BASE_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_Isolated_Pathways")

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a"
}

NEUTRAL_COLOR = "#777171"
CONTROL_COLOR = "#585656"

# Load Annotations
anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for highlight_pathway, highlight_color in MANUAL_COLORS.items():
    
    # Create a specific directory for this pathway
    pathway_folder_name = highlight_pathway.replace("/", "_").replace(" ", "_")
    pathway_output_dir = os.path.join(BASE_OUTPUT_DIR, pathway_folder_name)
    
    if not os.path.exists(pathway_output_dir):
        os.makedirs(pathway_output_dir)
        
    print(f"\nProcessing Pathway: {highlight_pathway}")

    for file_name in coord_files:
        df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
        
        # Merge metadata
        df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
        df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
        is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
        
        # Category setup
        df_plot['Display_Category'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3']).fillna("Unknown/Other")
        df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
        df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

        fig = go.Figure()

        # --- 3. KDE CONTOUR (Control Group) ---
        ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
        if not ctrl_df.empty:
            x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
            kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
            densities = kde(np.vstack([x_pts, y_pts]))
            levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
            
            x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
            y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
            X, Y = np.meshgrid(x_grid, y_grid)
            Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

            fig_tmp, ax_tmp = plt.subplots()
            cs = ax_tmp.contour(X, Y, Z, levels=levels)
            for i, segs in enumerate(cs.allsegs):
                for j, seg in enumerate(segs):
                    fig.add_trace(go.Scatter(
                        x=seg[:,0], y=seg[:,1], mode='lines',
                        line=dict(color=CONTROL_COLOR, width=1.5),
                        showlegend=False, hoverinfo='skip'
                    ))
            plt.close(fig_tmp)

        # --- 4. DATA POINTS (Layered) ---
        # First layer: Grey background points
        grey_df = df_plot[(df_plot['Display_Category'] != highlight_pathway) & (df_plot['Display_Category'] != 'Control Group')]
        
        fig.add_trace(go.Scatter(
            x=grey_df['UMAP1'], y=grey_df['UMAP2'], mode='markers',
            name="Other/Background",
            marker=dict(size=MARKER_SIZE, color=NEUTRAL_COLOR, opacity=0.3, line=dict(width=0.2, color='white')),
            text=grey_df['Treatment'],
            hoverinfo='skip'
        ))

        # Second layer: The Highlight Pathway
        target_df = df_plot[df_plot['Display_Category'] == highlight_pathway]
        if not target_df.empty:
            fig.add_trace(go.Scatter(
                x=target_df['UMAP1'], y=target_df['UMAP2'], mode='markers',
                name=highlight_pathway,
                marker=dict(size=MARKER_SIZE + 2,
                            color=highlight_color, 
                            opacity=1.0, 
                            line=dict(width=0.8, color='black')),
                text=target_df['Treatment'],
                hovertemplate="<b>%{text}</b><br>Path: " + highlight_pathway + "<extra></extra>"
            ))

        # --- 5. LAYOUT ---
        fig.update_layout(
            width=1400, height=900,
            template='plotly_white',
            showlegend=True,
            legend=dict(x=0.82, y=1, font=dict(size=LEGEND_FONT_SIZE)),
            margin=dict(l=100, r=350, t=80, b=100),
            xaxis=dict(
                title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                showgrid=False, showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH,
                ticks="outside", tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                constrain='domain', domain=[0, 0.75]
            ),
            yaxis=dict(
                title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                showgrid=False, showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH,
                ticks="outside", tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                scaleanchor="x", scaleratio=1, constrain='domain'
            )
        )

        base_name = f"Highlight_{pathway_folder_name}_{file_name.replace('.csv', '')}"
        fig.write_image(os.path.join(pathway_output_dir, f"{base_name}.png"), scale=PNG_RESOLUTION_SCALE)
        fig.write_image(os.path.join(pathway_output_dir, f"{base_name}.svg"))
        fig.write_html(os.path.join(pathway_output_dir, f"{base_name}.html"))

print("\nTask Complete. Each pathway has its own folder with isolated highlight plots.")

In [ ]:
#annotation with names

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 14           
MARKER_OPACITY = 0.8      
DOT_LABEL_SIZE = 24       
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

# Using existing coordinates folder from your first run
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_annotated_with_Names")

for d in [OUTPUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "unknown function": "#000000"  
}

# Load Annotations
anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Annotated Plot with Names: {file_name}...")
    
    # Load coordinates
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Merge and clean metadata
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    df_plot['Effective_Annotation'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3']).fillna("unknown function")
    df_plot['Display_Category'] = df_plot['Effective_Annotation']
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend Setup
    unique_in_data = df_plot['Display_Category'].unique()
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["unknown function"]
    
    color_lookup = MANUAL_COLORS.copy()
    auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
    for i, cat in enumerate(other_cats):
        color_lookup[cat] = auto_pal[i % len(auto_pal)]

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.2),
                    showlegend=False, hoverinfo='skip',
                    name='Control Group' if i == 0 and j == 0 else 'inner'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS WITH NAMES ---
    # --- 4. DATA POINTS WITH NAMES ---
    remaining_cats = [c for c in final_legend_order[::-1] if c != "Control Group"]
    for cat in remaining_cats:
        if cat not in df_plot['Display_Category'].values: continue
        sub_df = df_plot[df_plot['Display_Category'] == cat]
        
        # Hover logic
        hover_labels = []
        for _, r in sub_df.iterrows():
            # Added Well_ID to the hover string below
            txt = (f"<b>Treatment:</b> {r.get('Treatment','N/A')}<br>"
                   f"<b>Well ID:</b> {r.get('Well_ID','N/A')}<br>" 
                   f"<b>Category:</b> {cat}<br>"
                   f"<b>Timepoint:</b> {r.get('Timepoint','N/A')}")
            hover_labels.append(txt)

        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], y=sub_df['UMAP2'], 
            mode='markers+text',
            text=sub_df['Treatment'],
            textposition='top center',
            textfont=dict(size=DOT_LABEL_SIZE, color='black'),
            name=cat, legendgroup=cat, showlegend=False,
            marker=dict(size=MARKER_SIZE, color=color_lookup.get(cat, "#000000"),
                        opacity=MARKER_OPACITY, line=dict(width=0.4, color='white')),
            customdata=hover_labels,
            hovertemplate="%{customdata}<extra></extra>"
        ))

    # --- 5. FIXED SQUARE LAYOUT (70/30 SPLIT) ---
     # C) Final Layout (Consistent Square Plot Area)
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               # Positioned in the space created by the domain
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), # Increased right margin for legend
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False,
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            # REMOVE OR COMMENT OUT THIS LINE:
            # tickformat='.0f', 
            constrain='domain',
            domain=[0, 0.75]
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False,
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            # REMOVE OR COMMENT OUT THIS LINE:
            # tickformat='.0f',
            scaleanchor="x",
            scaleratio=1,
            constrain='domain'
        )
    )

    # Legend Override
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            name = trace.name if trace.name else ""
            if (name == cat or (cat == 'Control Group' and name.startswith('Control Group'))) and cat not in seen:
                trace.showlegend = True
                trace.name = cat
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    base_name = file_name.replace("Coords_", "UMAP_Names_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Annotated plots with Treatment names and 70/30 square layout are saved.")

In [ ]:
#annotation with names html zoom in als svg

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 26  
AXIS_TITLE_FONT_SIZE = 26 

LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 16           
MARKER_OPACITY = 0.8      
DOT_LABEL_SIZE = 27       
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

# Using existing coordinates folder from your first run
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_annotated_with_Names_ZOOM")

for d in [OUTPUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "unknown function": "#000000"  
}

# Load Annotations
anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Annotated Plot with Names: {file_name}...")
    
    # Load coordinates
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Merge and clean metadata
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    df_plot['Effective_Annotation'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3']).fillna("unknown function")
    df_plot['Display_Category'] = df_plot['Effective_Annotation']
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend Setup
    unique_in_data = df_plot['Display_Category'].unique()
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["unknown function"]
    
    color_lookup = MANUAL_COLORS.copy()
    auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
    for i, cat in enumerate(other_cats):
        color_lookup[cat] = auto_pal[i % len(auto_pal)]

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.2),
                    showlegend=False, hoverinfo='skip',
                    name='Control Group' if i == 0 and j == 0 else 'inner'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS WITH NAMES ---
    # --- 4. DATA POINTS WITH NAMES ---
    remaining_cats = [c for c in final_legend_order[::-1] if c != "Control Group"]
    for cat in remaining_cats:
        if cat not in df_plot['Display_Category'].values: continue
        sub_df = df_plot[df_plot['Display_Category'] == cat]
        
        # Hover logic
        hover_labels = []
        for _, r in sub_df.iterrows():
            # Added Well_ID to the hover string below
            txt = (f"<b>Treatment:</b> {r.get('Treatment','N/A')}<br>"
                   f"<b>Well ID:</b> {r.get('Well_ID','N/A')}<br>" 
                   f"<b>Category:</b> {cat}<br>"
                   f"<b>Timepoint:</b> {r.get('Timepoint','N/A')}")
            hover_labels.append(txt)

        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], y=sub_df['UMAP2'], 
            mode='markers+text',
            text=sub_df['Treatment'],
            textposition='top center',
            textfont=dict(size=DOT_LABEL_SIZE, color='black', style= 'italic'),
            name=cat, legendgroup=cat, showlegend=False,
            marker=dict(size=MARKER_SIZE, color=color_lookup.get(cat, "#000000"),
                        opacity=MARKER_OPACITY, line=dict(width=0.4, color='white')),
            customdata=hover_labels,
            hovertemplate="%{customdata}<extra></extra>"
        ))

    # --- 5. FIXED SQUARE LAYOUT (70/30 SPLIT) ---
     # C) Final Layout (Consistent Square Plot Area)
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               # Positioned in the space created by the domain
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), # Increased right margin for legend
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False,
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            # REMOVE OR COMMENT OUT THIS LINE:

            
            constrain='domain',
            domain=[0, 0.75]
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False,
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            # REMOVE OR COMMENT OUT THIS LINE:
            # tickformat='.0f',
            scaleanchor="x",
            scaleratio=1,
            constrain='domain'
        )
    )

    # Legend Override
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            name = trace.name if trace.name else ""
            if (name == cat or (cat == 'Control Group' and name.startswith('Control Group'))) and cat not in seen:
                trace.showlegend = True
                trace.name = cat
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    base_name = file_name.replace("Coords_", "UMAP_Names_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    config = {
    'toImageButtonOptions': {
        'format': 'svg',        # Changed from 'png' to 'svg'
        'filename': 'zoomed_cluster_export',
        'height': 900,
        'width': 1400,
        'scale': 1              # Scale is ignored for SVG since vectors are resolution-independent
    }
    }
    fig.write_html(
    os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")),
    config=config)




print("\nDone. Annotated plots with Treatment names and 70/30 square layout are saved.")

In [ ]:
#14 april with names, html zoom, other color

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 26  
AXIS_TITLE_FONT_SIZE = 26 

LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 16           
MARKER_OPACITY = 0.8      
DOT_LABEL_SIZE = 27       
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "14april_T0T1_annotated_with_Names_ZOOM")

for d in [OUTPUT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "unknown function": "#292727"  
}

# Load Annotations
anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Annotated Plot with Names: {file_name}...")
    
    # Load coordinates
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Merge and clean metadata
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    df_plot['Effective_Annotation'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3']).fillna("unknown function")
    df_plot['Display_Category'] = df_plot['Effective_Annotation']
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend and Color Logic
    unique_in_data = df_plot['Display_Category'].unique()
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["unknown function"]
    
    # Apply coloring logic: if not in MANUAL_COLORS, use black
    color_lookup = MANUAL_COLORS.copy()
    for cat in other_cats:
        color_lookup[cat] ="#292727"

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.2),
                    showlegend=False, hoverinfo='skip',
                    name='Control Group' if i == 0 and j == 0 else 'inner'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS WITH NAMES ---
    # Controls represented as contours, so they are excluded from point plotting
    remaining_cats = [c for c in final_legend_order[::-1] if c != "Control Group"]
    for cat in remaining_cats:
        if cat not in df_plot['Display_Category'].values: continue
        sub_df = df_plot[df_plot['Display_Category'] == cat]
        
        hover_labels = []
        for _, r in sub_df.iterrows():
            txt = (f"<b>Treatment:</b> {r.get('Treatment','N/A')}<br>"
                   f"<b>Well ID:</b> {r.get('Well_ID','N/A')}<br>" 
                   f"<b>Category:</b> {cat}<br>"
                   f"<b>Timepoint:</b> {r.get('Timepoint','N/A')}")
            hover_labels.append(txt)

        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], y=sub_df['UMAP2'], 
            mode='markers+text',
            text=sub_df['Treatment'],
            textposition='top center',
            textfont=dict(size=DOT_LABEL_SIZE, color='black', style='italic'),
            name=cat, legendgroup=cat, showlegend=False,
            marker=dict(size=MARKER_SIZE, 
                        color=color_lookup.get(cat, "#000000"),
                        opacity=MARKER_OPACITY, 
                        line=dict(width=0.4, color='white')),
            customdata=hover_labels,
            hovertemplate="%{customdata}<extra></extra>"
        ))

    # --- 5. FIXED SQUARE LAYOUT ---
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82, 
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False,
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            constrain='domain',
            domain=[0, 0.75]
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False,
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            scaleanchor="x",
            scaleratio=1,
            constrain='domain'
        )
    )

    # Legend Override
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            name = trace.name if trace.name else ""
            if (name == cat or (cat == 'Control Group' and name.startswith('Control Group'))) and cat not in seen:
                trace.showlegend = True
                trace.name = cat
                seen.add(cat)
                try:
                    trace.legendrank = final_legend_order.index(cat)
                except ValueError:
                    pass

    base_name = file_name.replace("Coords_", "UMAP_Names_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    
    config = {
        'toImageButtonOptions': {
            'format': 'svg',
            'filename': 'zoomed_cluster_export',
            'height': 900,
            'width': 1400,
            'scale': 1
        }
    }
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")), config=config)

print("\nDone. Zoomed plots with black coloring for unlisted pathways and square formatting are saved.")

In [ ]:
#target operon coloring

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS (KEEPING EXACTLY THE SAME) ---
AXIS_LINE_WIDTH = 3        
AXIS_TICK_WIDTH = 3        
AXIS_TICK_LEN = 8          
AXIS_TICK_FONT_SIZE = 24   
AXIS_TITLE_FONT_SIZE = 24  

LEGEND_FONT_SIZE = 14      
LEGEND_TITLE_SIZE = 16     

MARKER_SIZE = 8            
MARKER_OPACITY = 0.8       
PNG_RESOLUTION_SCALE = 7   

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")

COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_Operon_Contour")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Manual colors for common groups
MANUAL_COLORS = {
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "unknown function": "#000000",
}

# Load Annotations
anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Operon Contour Overlay for: {file_name}...")
    
    # Load coordinates
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Re-merge focusing on 'Target Operon'
    df_plot = df_coords.merge(anno_df[['Treatment', 'Target Operon']], on='Treatment', how='left')
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    # Category setup using Operon
    df_plot['Display_Category'] = df_plot['Target Operon'].fillna("unknown function")
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend Logic
    unique_in_data = df_plot['Display_Category'].unique()
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + other_cats + ["unknown function"]
    
    color_lookup = MANUAL_COLORS.copy()
    # Expanded palette for many operons
    auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24
    for i, cat in enumerate(other_cats):
        color_lookup[cat] = auto_pal[i % len(auto_pal)]

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    ctrl_df = df_plot[df_plot['Display_Category'] == 'Control Group']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.5),
                    showlegend=False, hoverinfo='skip',
                    name='Control Group' if i == 0 and j == 0 else 'inner'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS ---
    for cat in final_legend_order[::-1]:
        if cat not in df_plot['Display_Category'].values or cat == "Control Group": continue
        sub_df = df_plot[df_plot['Display_Category'] == cat]
        
        fig.add_trace(go.Scatter(
            x=sub_df['UMAP1'], y=sub_df['UMAP2'], mode='markers',
            name=cat, legendgroup=cat, showlegend=False,
            marker=dict(size=MARKER_SIZE, color=color_lookup.get(cat, "#000000"),
                        opacity=MARKER_OPACITY, line=dict(width=0.4, color='white')),
            text=sub_df['Treatment'],
            hovertemplate="<b>%{text}</b><br>Operon: " + str(cat) + "<extra></extra>"
        ))

    # --- 5. FIXED SQUARE LAYOUT (EXACT REPLICA OF YOUR REQUESTED STYLE) ---
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82, 
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), 
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] 
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", 
            scaleratio=1,
            constrain='domain'
        )
    )

    # Final legend visibility override
    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            if (trace.name == cat or (cat == 'Control Group' and trace.name == 'Control Group')) and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    base_name = file_name.replace("Coords_", "UMAP_Operon_")
    
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Operon plots generated with your exact layout specifications.")

In [ ]:
#mask overlay met controls als squares (iets te lange x as), als je niks van legende oet is het  ok hoor

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8          
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

MARKER_SIZE = 8            
MARKER_OPACITY = 0.8       
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP & AREA DATA LOADING ---
PROJECT_ROOT = r'E:\Thesis3april'
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Mask_Area")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

area_path = r'E:\Thesis3april\overlay\master_median_areas_per_site.csv'
area_df = pd.read_csv(area_path)
area_df['Well_ID'] = area_df['Plate'] + "_" + area_df['Well']
area_map = area_df.groupby('Well_ID')['Median_Area'].median().to_dict()

# --- 2. EXECUTION LOOP ---
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

for file_name in coord_files:
    print(f"Plotting Area Gradient (Colorbar Only) from: {file_name}...")
    
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    df_coords['Median_Area_Size'] = df_coords['Well_ID'].map(area_map)
    df_coords['Median_Area_Size'] = df_coords['Median_Area_Size'].fillna(df_coords['Median_Area_Size'].mean())
    
    color_min = df_coords['Median_Area_Size'].min()
    color_max = df_coords['Median_Area_Size'].quantile(0.99) 
    
    fig = px.scatter(
        df_coords, 
        x='UMAP1', 
        y='UMAP2', 
        color='Median_Area_Size',      
        symbol='Type', 
        hover_name='Treatment',
        hover_data={
            'Plate': True, 'Well_ID': True, 'Cell_Count': True,
            'Median_Area_Size': ':.2f'
        },
        range_color=[color_min, color_max],
        color_continuous_scale='Viridis',
        template='plotly_white'
    )
    
    fig.update_traces(marker=dict(size=MARKER_SIZE, opacity=MARKER_OPACITY, line=dict(width=0.5, color='DarkGrey')))
    
    # C) Final Layout (Consistent Square Plot Area)
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               # Positioned in the space created by the domain
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), # Increased right margin for legend
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, # L-shaped
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] # Reserves 25% of the width for legend/margin
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, # L-shaped
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", # Forces square aspect ratio
            scaleratio=1,
            constrain='domain'
        )
    )
    
    # Save Outputs
    base_name = file_name.replace("Coords_", "UMAP_Area_")
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)

print("\nDone. Plots generated with full-height gradient and matched fonts.")

In [ ]:
#mask overlay op contour (ok maar dan is de 16 erbij)

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS (UNTOUCHED) ---
AXIS_LINE_WIDTH = 4       
AXIS_TICK_WIDTH = 4       
AXIS_TICK_LEN = 10          
AXIS_TICK_FONT_SIZE = 30  
AXIS_TITLE_FONT_SIZE = 30 
LEGEND_FONT_SIZE = 35

MARKER_SIZE = 8            
MARKER_OPACITY = 0.8       
PNG_RESOLUTION_SCALE = 7  

FONT_FAMILY = "Arial"
AXIS_TITLE_FONT_SIZE = 30
AXIS_TICK_FONT_SIZE = 30



# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
# Using your specified coordinates directory
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_Contour_MaskArea")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Load the NEW Median Area Data (the one calculated from individual cells)
area_path = r'E:\Thesis3april\overlay\well_median_areas_from_cells.csv'
area_df = pd.read_csv(area_path)

# Create a mapping for both Area and Treatment based on Plate_Well
area_df['Well_ID'] = area_df['Plate'] + "_" + area_df['Well']
area_map = area_df.set_index('Well_ID')['Well_Median_Area'].to_dict()
# In case treatment is missing in coord files, we can also map it from our new master file
treatment_map = area_df.set_index('Well_ID')['Treatment'].to_dict()

coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Contour + Area Gradient for: {file_name}...")
    
    # Load coordinates
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Ensure Well_ID is present in coords (Plate_Well)
    # Add Area data from our new median file
    df_coords['Median_Area_Size'] = df_coords['Well_ID'].map(area_map)
    
    # Fill missing areas with mean if necessary
    if df_coords['Median_Area_Size'].isnull().any():
        df_coords['Median_Area_Size'] = df_coords['Median_Area_Size'].fillna(df_coords['Median_Area_Size'].mean())
    
    # Define color scale limits
    color_min = df_coords['Median_Area_Size'].quantile(0.05)
    color_max = df_coords['Median_Area_Size'].quantile(0.95)

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC (CONTROL GROUP ONLY) ---
    # We use the treatment column now reliably present in the data
    is_ctrl_mask = df_coords['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    ctrl_df = df_coords[is_ctrl_mask]
    
    # Create a dataframe for plotting that EXCLUDES controls
    plot_df = df_coords[~is_ctrl_mask].copy()
    
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        x_grid = np.linspace(df_coords['UMAP1'].min()-2, df_coords['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_coords['UMAP2'].min()-2, df_coords['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.5),
                    showlegend=False, hoverinfo='skip'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS (GRADIENT BASED ON WELL MEDIAN) ---
    fig.add_trace(go.Scatter(
        x=plot_df['UMAP1'], 
        y=plot_df['UMAP2'], 
        mode='markers',
        marker=dict(
            size=MARKER_SIZE,
            color=plot_df['Median_Area_Size'],
            colorscale='Viridis',
            cmin=color_min,
            cmax=color_max,
            opacity=MARKER_OPACITY,
            line=dict(width=0.4, color='white'),
            colorbar=dict(
                title=dict(
                    text="Median Area",
                    font=dict(size=AXIS_TITLE_FONT_SIZE)
                ),
                x=0.72,
                xanchor='left',
                thickness=30,
                lenmode='fraction',
                len=1.0,
                y=0.5,
                yanchor='middle',
                tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                ticks="outside",
                tickwidth=AXIS_TICK_WIDTH
            )
        ),
        text=plot_df['Treatment'],
        customdata=plot_df['Median_Area_Size'],
        hovertemplate="<b>%{text}</b><br>Area: %{customdata:.2f}<extra></extra>"
    ))

    # --- 5. FIXED SQUARE LAYOUT (70/30 SPLIT - UNTOUCHED) ---
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82, 
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), 
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] 
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", 
            scaleratio=1,
            constrain='domain'
        )
    )
    
    # Save Outputs
    base_name = file_name.replace("Coords_", "UMAP_ContourArea_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print(f"\nDone. Plots generated using individual-cell-derived medians from {area_path}")

In [ ]:
#AUC overlay

In [ ]:
import pandas as pd
import numpy as np
import os
import scipy.stats as st
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ==========================================
# 0. ADAPTABLE STYLE PARAMETERS
# ==========================================
FONT_FAMILY = "Arial"
AXIS_TITLE_FONT_SIZE = 30
AXIS_TICK_FONT_SIZE = 30

# Explicit Legend/Colorbar Font Sizes
LEGEND_FONT_SIZE = 35 
COLORBAR_TITLE_FONT_SIZE = 35

AXIS_LINE_WIDTH = 4
AXIS_TICK_WIDTH = 4
AXIS_TICK_LEN = 10

MARKER_SIZE = 8
MARKER_OPACITY = 0.8
PNG_RESOLUTION_SCALE = 7 

# ==========================================
# 1. SETUP
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_featureSelection", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "T0T1_AUC_Overlay")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# --- LOAD AUC DATA & MAPPER ---
auc_no_xylose_path = r'C:\Users\arnou\Documents\thesis\Resultaten\GrowthCurves\AUC_without_xylose.csv' 
auc_with_xylose_path = r'C:\Users\arnou\Documents\thesis\Resultaten\GrowthCurves\AUC_with_xylose.csv'

auc_no_xylose = pd.read_csv(auc_no_xylose_path)
auc_with_xylose = pd.read_csv(auc_with_xylose_path)

map_no_xylose = dict(zip(auc_no_xylose['Gene_target'], auc_no_xylose['AUC']))
map_with_xylose = dict(zip(auc_with_xylose['Gene_target'], auc_with_xylose['AUC']))

def assign_auc(row):
    treatment = str(row['Treatment'])
    plate = str(row.get('Plate', ''))
    if "_T0" in plate:
        return map_no_xylose.get(treatment, np.nan)
    else:
        return map_with_xylose.get(treatment, np.nan)

coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# ==========================================
# 2. EXECUTION LOOP
# ==========================================
for file_name in coord_files:
    print(f"Generating AUC Overlay for: {file_name}...")
    
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    
    # Mapping logic
    df_coords['AUC'] = df_coords.apply(assign_auc, axis=1)
    df_coords['Type'] = df_coords['Treatment'].apply(
        lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant'
    )
    
    # NEW: Filter points so we only plot "Mutant" dots
    # Controls will be represented ONLY by the contour
    df_valid = df_coords[(df_coords['AUC'].notna()) & (df_coords['Type'] == 'Mutant')].copy()
    df_missing = df_coords[(df_coords['AUC'].isna()) & (df_coords['Type'] == 'Mutant')].copy()
    
    color_min = df_coords['AUC'].quantile(0.95) # Keep scale consistent with full range if desired
    color_max = df_coords['AUC'].quantile(0.95)

    fig = go.Figure()

    # --- KDE CONTOUR (CONTROL GROUP) ---
    # We still use the full df_coords to find the controls for the contour
    ctrl_df = df_coords[df_coords['Type'] == 'Control']
    if not ctrl_df.empty:
        x_pts, y_pts = ctrl_df['UMAP1'].values, ctrl_df['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        
        x_grid = np.linspace(df_coords['UMAP1'].min()-2, df_coords['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_coords['UMAP2'].min()-2, df_coords['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for segs in cs.allsegs:
            for seg in segs:
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.5),
                    showlegend=False, hoverinfo='skip'
                ))
        plt.close(fig_tmp)

    # --- DATA TRACES (Mutants Only) ---
    # Missing points (Grey)
    if not df_missing.empty:
        fig.add_trace(go.Scatter(
            x=df_missing['UMAP1'], y=df_missing['UMAP2'],
            mode='markers',
            marker=dict(color='lightgrey', size=MARKER_SIZE, opacity=0.4, line=dict(width=0.3, color='white')),
            name='No AUC Data',
            text=df_missing['Treatment'],
            hovertemplate="<b>%{text}</b><br>AUC: N/A<extra></extra>"
        ))

    # Valid points (AUC Gradient)
    if not df_valid.empty:
        fig.add_trace(go.Scatter(
            x=df_valid['UMAP1'], y=df_valid['UMAP2'],
            mode='markers',
            marker=dict(
                size=MARKER_SIZE,
                color=df_valid['AUC'],
                colorscale='Viridis',
                cmin=color_min, cmax=color_max,
                opacity=MARKER_OPACITY,
                line=dict(width=0.4, color='white'),
                colorbar=dict(
                    title=dict(
                        text="AUC", 
                        font=dict(size=COLORBAR_TITLE_FONT_SIZE, family=FONT_FAMILY)
                    ),
                    x=0.72, xanchor='left', thickness=30,
                    lenmode='fraction', len=1.0,
                    y=0.5, yanchor='middle',
                    tickfont=dict(size=LEGEND_FONT_SIZE, family=FONT_FAMILY),
                    ticks="outside", tickwidth=AXIS_TICK_WIDTH
                )
            ),
            text=df_valid['Treatment'],
            customdata=df_valid['AUC'],
            hovertemplate="<b>%{text}</b><br>AUC: %{customdata:.4f}<extra></extra>"
        ))

    # --- SQUARE LAYOUT ---
    fig.update_layout(
        width=1400, 
        height=900,
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82, 
            y=1, 
            xanchor='left', 
            yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100), 
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] 
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", 
            scaleratio=1,
            constrain='domain'
        )
    )

    # --- SAVE OUTPUTS ---
    base_name = file_name.replace("Coords_", "UMAP_AUC_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. AUC dots for controls removed; only contours remain.")

In [ ]:
#andere groeikaraktereistieken erop plotten kan, maar dan zo ik ook max od, max peak prominence met en zonder xylose moeten berkenen
#(eventueel nog doen als nodig...), maar die groeikarakterisitken zeiden nu ook niet zo heel veel

In [ ]:
#antibiotics#
#######






#

In [ ]:
#aggregation antibiotics (dit run ik niet 3 april)

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
PLATES = ["PLATE6_T1","PLATE7_T1"]

# Paths for the three separate outputs
OUTPUT_CSV_MEDIAN = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_median.csv")
OUTPUT_CSV_MEAN   = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_mean.csv")
OUTPUT_CSV_STD    = os.path.join(PROJECT_ROOT,"31march", "antibiotics_aggregated_wells_std.csv")

CELL_COUNT_THRESHOLD = 0  
TREATMENT_COL = "Treatment"

all_plates_median = []
all_plates_mean = []
all_plates_std = []
all_cell_counts = [] 

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT, "features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT, "metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    # --- AGGREGATION & THRESHOLDING STEP ---
    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_cell_count = all_cells_in_well.shape[0]
        all_cell_counts.append(well_cell_count)

        if well_cell_count >= CELL_COUNT_THRESHOLD:
            # Calculate aggregations
            well_median = np.median(all_cells_in_well, axis=0)
            well_mean   = np.mean(all_cells_in_well, axis=0)
            well_std    = np.std(all_cells_in_well, axis=0)
            
            base_info = {
                "Plate": plate_id, 
                "Well_ID": well_id, 
                "Treatment": well_to_treatment[well_id],
                "Cell_Count": well_cell_count
            }
            
            # Efficiently map feature indices to values
            feat_cols = {idx: val for idx, val in enumerate(well_median)}
            all_plates_median.append({**base_info, **feat_cols})
            
            feat_cols_mean = {idx: val for idx, val in enumerate(well_mean)}
            all_plates_mean.append({**base_info, **feat_cols_mean})
            
            feat_cols_std = {idx: val for idx, val in enumerate(well_std)}
            all_plates_std.append({**base_info, **feat_cols_std})

# Convert to DataFrames
df_median = pd.DataFrame(all_plates_median)
df_mean   = pd.DataFrame(all_plates_mean)
df_std    = pd.DataFrame(all_plates_std)

# Helper function to reorder columns consistently
def reorder_cols(df):
    if df.empty: return df
    meta_cols = ["Plate", "Well_ID", "Treatment", "Cell_Count"]
    feat_cols = sorted([c for c in df.columns if c not in meta_cols])
    return df[meta_cols + feat_cols]

df_median = reorder_cols(df_median)
df_mean   = reorder_cols(df_mean)
df_std    = reorder_cols(df_std)

print(f"\nAggregation complete.")

# ==========================================
# 2. SAVE TO CSVs
# ==========================================
df_median.to_csv(OUTPUT_CSV_MEDIAN, index=False)
df_mean.to_csv(OUTPUT_CSV_MEAN, index=False)
df_std.to_csv(OUTPUT_CSV_STD, index=False)

print(f"Median data saved: {OUTPUT_CSV_MEDIAN}")
print(f"Mean data saved:   {OUTPUT_CSV_MEAN}")
print(f"Std Dev data saved: {OUTPUT_CSV_STD}")

In [ ]:
#min5 wells antiiobitcs (maar is niet nodig)

In [ ]:
import pandas as pd
import os

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_DIR = os.path.join(PROJECT_ROOT, "31march")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "31march_filtered")

# Create the directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. DEFINE YOUR NEW THRESHOLD
STRICT_THRESHOLD = 5 

# List of the aggregation files you created in the previous step
FILES_TO_FILTER = [
    "antibiotics_aggregated_wells_median.csv",
    "antibiotics_aggregated_wells_mean.csv",
    "antibiotics_aggregated_wells_std.csv"
]

# 3. PROCESSING LOOP
for file_name in FILES_TO_FILTER:
    file_path = os.path.join(INPUT_DIR, file_name)
    
    if not os.path.exists(file_path):
        print(f"Skipping {file_name}: File not found.")
        continue

    print(f"\n--- Processing {file_name} ---")
    df = pd.read_csv(file_path)

    # Identify the wells to keep and remove
    # We use 'Cell_Count' which we added during the aggregation step
    filtered_df = df[df['Cell_Count'] >= STRICT_THRESHOLD].copy()
    removed_df = df[df['Cell_Count'] < STRICT_THRESHOLD].copy()

    # 4. REPORT
    print(f"Original wells: {len(df)}")
    print(f"Wells kept:     {len(filtered_df)}")
    print(f"Wells removed:  {len(removed_df)}")

    if not removed_df.empty and "median" in file_name:
        # Just show the list once (for the median file) to avoid clutter
        print(f"\nExample of removed wells (Count < {STRICT_THRESHOLD}):")
        print(removed_df[['Plate', 'Well_ID', 'Treatment', 'Cell_Count']].head(10).to_string(index=False))

    # 5. SAVE DATA
    # Renaming the output to include the 'min5' suffix
    output_name = file_name.replace(".csv", "_min5.csv")
    output_path = os.path.join(OUTPUT_DIR, output_name)
    filtered_df.to_csv(output_path, index=False)
    
    print(f"Filtered data saved to: {output_path}")

print("\nAll filtering tasks complete!")

In [ ]:
#zelf die paar slechte verwijderd in de juist ifle

In [ ]:
#antibiotics alone median without feature selection

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# ==========================================
# 0. ADAPTABLE STYLE PARAMETERS
# ==========================================
FONT_FAMILY = "Arial"
AXIS_LINE_WIDTH = 3
AXIS_TICK_WIDTH = 3
AXIS_TICK_LEN = 8
AXIS_TICK_FONT_SIZE = 24
AXIS_TITLE_FONT_SIZE = 24
LEGEND_FONT_SIZE = 24
MARKER_SIZE = 10 
MARKER_OPACITY = 0.85
PNG_RESOLUTION_SCALE = 2

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", 'antibiotics_aggregated_wells_median_min5_juist.csv')
df = pd.read_csv(file_path)
df.columns = [str(c) for c in df.columns]

# Main nested folder
BASE_OUTPUT = os.path.join(PROJECT_ROOT, "plots8april", "AntibioticsMedian_withoutfeatureselction_12april")
os.makedirs(BASE_OUTPUT, exist_ok=True)

# Define the MoA Mapping
moa_map = {
    'vancomycin': 'Cell Wall', 'cefalexin': 'Cell Wall', 'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA', 'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 'kanamycin': 'Ribosome', 'tetracycline': 'Ribosome',
    'LB (control)': 'Control'
}

# ==========================================
# 2. UNIFIED METADATA PARSING
# ==========================================
def parse_all_metadata(row):
    treat = str(row['Treatment'])
    plate_raw = str(row['Plate'])
    
    is_control = any(ctrl in treat.lower() for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        antibiotic = "LB (control)"
        moa = "Control"
        conc = "LB (control)"
        time = "LB (control)"
    else:
        antibiotic = treat.split('_')[0]
        moa = moa_map.get(antibiotic, "Other")
        conc = "10 MIC" if "10" in treat else ("50 MIC" if "50" in treat else "Unknown")
        if treat.endswith('_1'): time = "T1"
        elif treat.endswith('_2'): time = "T2"
        else: time = "T0"
    
    clean_plate = "PLATE6" if "PLATE6" in plate_raw.upper() else "PLATE7"
    
    return pd.Series([antibiotic, moa, conc, time, clean_plate])

df[['Antibiotic', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']] = df.apply(parse_all_metadata, axis=1)

# ==========================================
# 3. DEFINE CHANNEL SUBSETS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. MASTER PLOTTING FUNCTION
# ==========================================
def run_batch_umap(overlay_name, target_col, folder_name, palette):
    # This creates the subfolder WITHIN the Antibiotics folder
    output_dir = os.path.join(BASE_OUTPUT, folder_name)
    os.makedirs(output_dir, exist_ok=True)
    
    unique_items = sorted(df[target_col].unique())
    control_label = "LB (control)" if target_col != "MOA" else "Control"
    others = [i for i in unique_items if i != control_label]
    
    color_map = {item: palette[i % len(palette)] for i, item in enumerate(others)}
    color_map[control_label] = "#000000"

    for config in plot_configs:
        if not config['indices']: continue
        print(f"Processing {overlay_name}: {config['name']}...")
        
        X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
        reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42)
        embedding = reducer.fit_transform(X_scaled)
        
        df_plot = df.copy()
        df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
        
        fig = go.Figure()
        ordered_groups = [control_label] + sorted(others)

        for group in ordered_groups:
            sub_df = df_plot[df_plot[target_col] == group]
            if sub_df.empty: continue
            
            is_ctrl = (group == control_label)
            
            fig.add_trace(go.Scatter(
                x=sub_df['UMAP1'], y=sub_df['UMAP2'],
                mode='markers',
                name=group,
                marker=dict(
                    symbol='circle',
                    size=12 if is_ctrl else MARKER_SIZE,
                    color=color_map[group],
                    opacity=1.0 if is_ctrl else MARKER_OPACITY,
                    line=dict(width=0.8, color='white')
                ),
                text=sub_df['Treatment'],
                hovertemplate=f"<b>%{{text}}</b><br>{overlay_name}: {group}<extra></extra>"
            ))

        fig.update_layout(
            template='plotly_white', width=1350, height=900,
            margin=dict(l=100, r=50, b=100, t=80), 
            font=dict(family=FONT_FAMILY),
            legend=dict(
                font=dict(size=LEGEND_FONT_SIZE),
                title=dict(text=overlay_name, font=dict(size=LEGEND_FONT_SIZE)),
                x=0.65, y=1, xanchor='left', yanchor='top',
                itemsizing='constant', traceorder="normal"
            ),
            xaxis=dict(
                title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 0.6], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE)
            ),
            yaxis=dict(
                title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 1], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                scaleanchor="x", scaleratio=1
            )
        )

        base_fn = f"UMAP_{overlay_name}_{config['name']}"
        fig.write_image(os.path.join(output_dir, f"{base_fn}.png"), scale=PNG_RESOLUTION_SCALE)
        fig.write_image(os.path.join(output_dir, f"{base_fn}.svg"))
        fig.write_html(os.path.join(output_dir, f"{base_fn}.html"))

# ==========================================
# 5. RUN ALL BATCHES
# ==========================================
tasks = [
    ("Antibiotic", "Antibiotic", "Antibiotics_Mean", px.colors.qualitative.Dark24),
    ("MOA", "MOA", "Antibiotics_MOA", px.colors.qualitative.Bold),
    ("Concentration", "Concentration", "Antibiotics_Concentration", px.colors.qualitative.Set1),
    ("Timepoint", "Timepoint", "Antibiotics_Timepoints", px.colors.qualitative.Vivid),
    ("Plate", "Plate_Clean", "Antibiotics_Plates", px.colors.qualitative.Safe)
]

for name, col, folder, pal in tasks:
    run_batch_umap(name, col, folder, pal)

print(f"\nProcessing complete. All subfolders are located in: {BASE_OUTPUT}")

In [ ]:
#antibiotics with knocdkwon plotted without feature seleciton

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.colors as mc
import colorsys

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3         
AXIS_TICK_WIDTH = 3         
AXIS_TICK_LEN = 8           
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 24     
LEGEND_SYMBOL_SIZE = 14   
HEADER_SYMBOL_SIZE = 14   

MARKER_SIZE = 8            
MARKER_OPACITY = 0.8      

# Helper function to lighten/darken colors
def adjust_color(color, amount=0.5):
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return mc.to_hex(colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2]))

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5_metP67.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antibiticsmedianwithknockdown_nofeatureselection_plates_12april")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1","PLATE3_T0","PLATE3_T1",
                   "PLATE4_T0","PLATE4_T1","PLATE5_T0","PLATE5_T1", "PLATE6_T1", "PLATE7_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl10"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

# --- 2. COLOR MAPPING ---
# I mapped these based on the order in your original list. 
# You can now change the hex codes for P6 and P7 directly here.
plate_color_map = {
    "P1_no_xyl": '#1f77b4', "P1_xyl5": '#ff7f0e',
    "P2_no_xyl": '#2ca02c', "P2_xyl5": '#d62728',
    "P3_no_xyl": '#9467bd', "P3_xyl5": '#8c564b',
    "P4_no_xyl": '#e377c2', "P4_xyl5": '#7f7f7f',
    "P5_no_xyl": '#bcbd22', "P5_xyl5": '#17becf',
    "P6_xyl5":   "#572A43", 
    "P7_xyl5":   "#8FDE4A"  
}

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_coords = df[['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- SAVE COORDINATES ---
    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    df_coords.to_csv(os.path.join(COORD_DIR, f"{base_name}_coords.csv"), index=False)

    fig = go.Figure()

    # --- UPDATED LEGEND HEADERS ---
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no-sgRNA', showlegend=True))
    
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True))

    conditions = sorted(df_coords['Plate_Display'].unique())
    for i, condition in enumerate(conditions):
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        
        # Pull color from map, default to grey if not specified
        base_color = plate_color_map.get(condition, '#7f7f7f')
        
        mutant_color = adjust_color(base_color, amount=1.2) 
        control_color = adjust_color(base_color, amount=0.7) 
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            color = mutant_color if t_type == 'Mutant' else control_color
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(symbol='circle' if t_type == 'Mutant' else 'square',
                            size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                            line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)),
                text=sub_data['Treatment'], hoverinfo='text+name'))

    # --- UPDATED LAYOUT FOR CONSISTENT SQUARE PLOT AREA ---
    fig.update_layout(
        width=1400,  
        height=900,  
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               
            y=1, 
            xanchor='left',
            yanchor='top',
        ),
        
        margin=dict(l=100, r=350, t=80, b=100),

        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] 
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", 
            scaleratio=1,
            constrain='domain'
        )
    )

    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

print("\nDone. Plots and coordinates generated successfully.")

In [ ]:
#new antibitoics files

In [ ]:
#feature selection met antibioticaplaten erbij       (die paar slechte verwijdert...)

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'

INPUT_CSV = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5_metP67.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT,"files")
CONTROL_LABEL = "no_sgRNA" 

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Loading raw data...")
df_raw_full = pd.read_csv(INPUT_CSV)
df_raw_full.columns = [str(c) for c in df_raw_full.columns]

# --- STEP: FILTER AND VIRTUAL MAPPING ---
df_raw = df_raw_full[df_raw_full['Plate'].str.contains('T0|T1')].copy()
df_raw['Virtual_TP'] = df_raw['Plate'].str[-2:]
mask = df_raw['Plate'].isin(['PLATE6_T1', 'PLATE7_T1'])
df_raw.loc[mask, 'Virtual_TP'] = 'T0'

metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Virtual_TP']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def get_channel_counts(features):
    """Counts features per channel based on numeric index ranges."""
    counts = {f"Channel_{i}": 0 for i in range(1, 6)}
    counts['Unknown/Out of Range'] = 0
    
    for f in features:
        try:
            val = int(f)
            if 0 <= val < 1280:
                counts["Channel_1"] += 1
            elif 1280 <= val < 2560:
                counts["Channel_2"] += 1
            elif 2560 <= val < 3840:
                counts["Channel_3"] += 1
            elif 3840 <= val < 5120:
                counts["Channel_4"] += 1
            elif 5120 <= val < 6400:
                counts["Channel_5"] += 1
            else:
                counts['Unknown/Out of Range'] += 1
        except ValueError:
            counts['Unknown/Out of Range'] += 1
    return counts

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby(['Plate', 'Virtual_TP'])[features].median().reset_index()

    tp_batch_noises = []
    for tp in plate_medians['Virtual_TP'].unique():
        tp_subset = plate_medians[plate_medians['Virtual_TP'] == tp][features]
        if len(tp_subset) > 1:
            noise = tp_subset.std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features, len(to_drop)

# ==========================================
# EXECUTION PIPELINE
# ==========================================

print(f"\nStarting with {len(feature_cols)} total features.")

# --- STEP 0: GLOBAL VARIANCE ---
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
removed_s0 = len(feature_cols) - len(active_features)
print(f"Step 0 (Variance Filter): Removed {removed_s0}, {len(active_features)} remaining.")

# --- STEP 1: WITHIN-PLATE ---
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
removed_s1 = len(active_features) - len(step1_features)
print(f"Step 1 (Consistency):    Removed {removed_s1}, {len(step1_features)} remaining.")
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE ---
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
removed_s2 = len(step1_features) - len(step2_features)
print(f"Step 2 (Stability):      Removed {removed_s2}, {len(step2_features)} remaining.")
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY ---
final_feature_list, removed_s3 = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)
print(f"Step 3 (Redundancy):     Removed {removed_s3}, {len(final_feature_list)} remaining.")

# ==========================================
# FINAL REPORT & SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list].drop(columns=['Virtual_TP'])
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_median_T0_T1_met67_9april.csv")
df_final.to_csv(output_path, index=False)

channel_breakdown = get_channel_counts(final_feature_list)

print("\n" + "="*40)
print(f"FINAL FEATURE BREAKDOWN PER CHANNEL")
print("="*40)
for ch, count in channel_breakdown.items():
    if count > 0 or "Channel" in ch:
        print(f"{ch:10} : {count} features")

print(f"\nSaved results to: {output_path}")
print("="*40)

In [ ]:
#antibiotica plotten op de knockdown, maak de layout nog mooier na Bartinfo over ocnsitnet layout +  hoe juist visualiseren

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---



PROJECT_ROOT = r'E:\Thesis3april'




file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_median_T0_T1_met67_9april.csv")
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "vettedcellcounts_antibi_T0_T1_met67color_9april")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading data...")
df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

# --- 2. GLOBAL VARIANCE FILTER ---
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c.isdigit()]
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
df = df_raw[metadata_cols + active_features].copy()

# --- 3. MERGE & CATEGORIZATION ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Identify groups
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
is_special_plate = df['Plate'].str.contains('PLATE6|PLATE7', case=False, na=False)

# NEW: Create a simplified Treatment name (everything before the first underscore)
# e.g., "vancomycin_50" becomes "vancomycin"
df['Treatment_Base'] = df['Treatment'].str.split('_').str[0]

# CATEGORIZATION LOGIC
df['Display_Category'] = df['Effective_Annotation']

# If it's PLATE6/7 and NOT a control, use the BASE Treatment name for coloring
df.loc[is_special_plate & ~is_control, 'Display_Category'] = df['Treatment_Base']

# Standard overrides
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control) & (~is_special_plate), 'Display_Category'] = 'Baseline (T0)'

# --- 4. GOLDEN ANGLE COLOR MAPPING ---
exclude = ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in exclude])
num_cats = len(all_cats)

if num_cats > 0:
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

color_map['no_sgrna'] = '#EBEBEB'
color_map['Baseline (T0)'] = '#B0B0B0'
color_map['Unknown/Other'] = '#222222'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Define shapes
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'
    df.loc[is_special_plate, 'Point_Shape'] = 'diamond'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square", "diamond": "diamond"},
        hover_name='Treatment', # Keep full name in hover
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # Trace/Legend clean up
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Styling
    fig.update_traces(marker=dict(opacity=0.9, line=dict(width=0.5, color='white'))) 
    
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=10, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square'))
    
    # Apply Large Diamond styling (colored by Treatment_Base)
    fig.update_traces(
        marker=dict(size=16, line=dict(width=1.5, color='black')), 
        selector=dict(marker_symbol='diamond')
    )
    
    fig.update_layout(
        width=1400, height=900,
        legend_title_text='Group / Compound (Diamonds)',
        xaxis=dict(title="UMAP 1", showline=True, linewidth=2, linecolor='black', showgrid=False),
        yaxis=dict(title="UMAP 2", showline=True, linewidth=2, linecolor='black', showgrid=False)
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))

print(f"Done. Plate 6/7 points are diamonds colored by compound name.")

In [ ]:
#antibiotics met knockdown same code as pure knockdown, 12 april

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.colors as mc
import colorsys

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3          
AXIS_TICK_WIDTH = 3          
AXIS_TICK_LEN = 8            
AXIS_TICK_FONT_SIZE = 24   
AXIS_TITLE_FONT_SIZE = 24  

LEGEND_FONT_SIZE = 24      
LEGEND_SYMBOL_SIZE = 14    
HEADER_SYMBOL_SIZE = 14    

MARKER_SIZE = 8             
MARKER_OPACITY = 0.8       

# Helper function to lighten/darken colors
def adjust_color(color, amount=0.5):
    try:
        c = mc.cnames[color]
    except:
        c = color
    c = colorsys.rgb_to_hls(*mc.to_rgb(c))
    return mc.to_hex(colorsys.hls_to_rgb(c[0], max(0, min(1, amount * c[1])), c[2]))

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_median_T0_T1_met67_9april.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antibiticsmedianwithknockdown_plates_12april")
COORD_DIR = os.path.join(OUTPUT_DIR, "coordinates")

for d in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

df = pd.read_csv(file_path)
df.columns = df.columns.astype(str)

SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE2_T0","PLATE2_T1","PLATE3_T0","PLATE3_T1",
                   "PLATE4_T0","PLATE4_T1","PLATE5_T0","PLATE5_T1", "PLATE6_T1", "PLATE7_T1"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

time_map = {"T0": "no_xyl", "T1": "xyl5", "T2": "xyl10"}
unique_plates = sorted(df['Plate'].unique())
plate_rename_map = {}
for p in unique_plates:
    new_name = p.replace("PLATE", "P")
    for t_old, t_new in time_map.items():
        if t_old in new_name:
            new_name = new_name.replace(t_old, t_new)
    plate_rename_map[p] = new_name

df['Plate_Display'] = df['Plate'].map(plate_rename_map)

# --- 2. COLOR MAP SELECTION ---
plate_color_map = {
    "P1_no_xyl": '#1f77b4', "P1_xyl5": '#ff7f0e',
    "P2_no_xyl": '#2ca02c', "P2_xyl5": '#d62728',
    "P3_no_xyl": '#9467bd', "P3_xyl5": '#8c564b',
    "P4_no_xyl": '#e377c2', "P4_xyl5": '#7f7f7f',
    "P5_no_xyl": '#bcbd22', "P5_xyl5": '#17becf',
    "P6_xyl5":   "#572A43", 
    "P7_xyl5":   "#8FDE4A"  
}

# --- 3. DEFINE CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. RUN UMAP LOOP ---
for config in plot_configs:
    valid_indices = [idx for idx in config['indices'] if idx in df.columns]
    if not valid_indices: continue
    
    X_scaled = StandardScaler().fit_transform(df[valid_indices].values)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_coords = df[['Plate', 'Plate_Display', 'Well_ID', 'Treatment', 'Type', 'Cell_Count']].copy()
    df_coords['UMAP1'], df_coords['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- SAVE COORDINATES ---
    base_name = f"UMAP_{config['name'].replace(' ', '_')}"
    df_coords.to_csv(os.path.join(COORD_DIR, f"{base_name}_coords.csv"), index=False)

    fig = go.Figure()

    # --- UPDATED LEGEND HEADERS ---
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(symbol='square-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='no-sgRNA', showlegend=True))
    
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
        marker=dict(symbol='circle-open', color='black', size=HEADER_SYMBOL_SIZE),
        name='Mutant', showlegend=True))

    conditions = sorted(df_coords['Plate_Display'].unique())
    for condition in conditions:
        cond_data = df_coords[df_coords['Plate_Display'] == condition]
        
        base_color = plate_color_map.get(condition, '#7f7f7f')
        mutant_color = adjust_color(base_color, amount=1.2) 
        control_color = adjust_color(base_color, amount=0.7) 
        
        for t_type in ['Mutant', 'Control']:
            sub_data = cond_data[cond_data['Type'] == t_type]
            if sub_data.empty: continue
            color = mutant_color if t_type == 'Mutant' else control_color
            
            # Create hover text strings combining Treatment and Well_ID
            hover_text = sub_data['Treatment'] + "<br>Well: " + sub_data['Well_ID']
            
            fig.add_trace(go.Scatter(
                x=sub_data['UMAP1'], y=sub_data['UMAP2'],
                mode='markers', name=condition,
                showlegend=(t_type == 'Mutant'), 
                legendgroup=condition,
                marker=dict(symbol='circle' if t_type == 'Mutant' else 'square',
                            size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY,
                            line=dict(width=1, color='black') if t_type == 'Control' else dict(width=0)),
                text=hover_text, 
                hoverinfo='text+name'))

    # --- UPDATED LAYOUT FOR CONSISTENT SQUARE PLOT AREA ---
    fig.update_layout(
        width=1400,  
        height=900,  
        autosize=False,
        template='plotly_white',
        font=dict(family="Arial"),
        
        legend=dict(
            itemsizing='constant', 
            font=dict(size=LEGEND_FONT_SIZE),
            x=0.82,               
            y=1, 
            xanchor='left',
            yanchor='top',
        ),
        
        margin=dict(l=100, r=350, t=80, b=100),

        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            constrain='domain',
            domain=[0, 0.75] 
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            mirror=False, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            tickformat='.0f',
            scaleanchor="x", 
            scaleratio=1,
            constrain='domain'
        )
    )

    fig.write_html(os.path.join(OUTPUT_DIR, f"{base_name}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.svg"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{base_name}.png"), scale=7)
    
    print(f"Finished {config['name']}")

print("\nDone. Plots and coordinates generated successfully.")

In [ ]:
#all antibiotics pltoted black on top of the normal knockdown pathways (12april)

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 3       
AXIS_TICK_WIDTH = 3       
AXIS_TICK_LEN = 8         
AXIS_TICK_FONT_SIZE = 24  
AXIS_TITLE_FONT_SIZE = 24 

LEGEND_FONT_SIZE = 14     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 8           # Size for standard circles
SPECIAL_MARKER_SIZE = 10  # Size for Plate 6 & 7 Diamonds/Squares
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antibiticsmedianwithknockdown_plates_12april", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antibiticsmedianwithknockdown_normalcode")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "biosynthesis of isoprenoids": "#7f7f7f",     
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "unknown function": "#000000",
}

anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Contour Overlay for: {file_name}...")
    
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    # Category setup
    df_plot['Display_Category'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3']).fillna("unknown function")
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend Logic
    unique_in_data = df_plot['Display_Category'].unique()
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["unknown function"]
    
    color_lookup = MANUAL_COLORS.copy()
    auto_pal = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
    for i, cat in enumerate(other_cats):
        color_lookup[cat] = auto_pal[i % len(auto_pal)]

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    mask_not_special = ~df_plot['Plate'].astype(str).str.contains('PLATE6|PLATE7')
    ctrl_df_for_kde = df_plot[(df_plot['Display_Category'] == 'Control Group') & mask_not_special]

    if not ctrl_df_for_kde.empty and len(ctrl_df_for_kde) > 5:
        x_pts, y_pts = ctrl_df_for_kde['UMAP1'].values, ctrl_df_for_kde['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.5),
                    showlegend=False, hoverinfo='skip'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS (Layered by Z-Order) ---
    for is_special_pass in [False, True]:
        for cat in final_legend_order[::-1]:
            if cat not in df_plot['Display_Category'].values: continue
            
            sub_df = df_plot[df_plot['Display_Category'] == cat]
            
            plot_x, plot_y, plot_txt, symbols, colors, sizes = [], [], [], [], [], []
            
            for _, row in sub_df.iterrows():
                row_is_ctrl = str(row['Treatment']).lower() in ["no_sgrna", "nosgrna"]
                row_plate_str = str(row['Plate'])
                row_plate_special = "PLATE6" in row_plate_str or "PLATE7" in row_plate_str
                
                if row_plate_special != is_special_pass:
                    continue

                if row_is_ctrl and not row_plate_special:
                    continue 
                
                plot_x.append(row['UMAP1'])
                plot_y.append(row['UMAP2'])
                plot_txt.append(row['Treatment'])
                
                if row_plate_special:
                    # Special size and symbol for Plate 6/7
                    sizes.append(SPECIAL_MARKER_SIZE)
                    if row_is_ctrl:
                        symbols.append("square")
                    else:
                        symbols.append("diamond")
                    colors.append("black")
                else:
                    # Default size and symbol
                    sizes.append(MARKER_SIZE)
                    symbols.append("circle")
                    colors.append(color_lookup.get(cat, "#000000"))

            if not plot_x: continue

            fig.add_trace(go.Scatter(
                x=plot_x, y=plot_y, mode='markers',
                name=cat, legendgroup=cat, showlegend=False,
                marker=dict(
                    size=sizes, 
                    color=colors,
                    symbol=symbols,
                    opacity=MARKER_OPACITY, 
                    line=dict(width=0.4, color='white')
                ),
                text=plot_txt,
                hovertemplate="<b>%{text}</b><br>Cat: " + cat + "<extra></extra>"
            ))

    # --- 5. FIXED SQUARE LAYOUT ---
    fig.update_layout(
        width=1400, height=900, autosize=False,
        template='plotly_white', font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', font=dict(size=LEGEND_FONT_SIZE),
            x=0.82, y=1, xanchor='left', yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE), tickformat='.0f',
            constrain='domain', domain=[0, 0.75]
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE), tickformat='.0f',
            scaleanchor="x", scaleratio=1, constrain='domain'
        )
    )

    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            if trace.name == cat and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    base_name = file_name.replace("Coords_", "UMAP_Contour_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Contours and standard points finished with Plate 6/7 oversized on top.")

In [ ]:
#all antibiotics pltoted black on top of the normal knockdown pathways (14april) andere colors

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 4       
AXIS_TICK_WIDTH = 4       
AXIS_TICK_LEN = 10         
AXIS_TICK_FONT_SIZE = 30  
AXIS_TITLE_FONT_SIZE = 30

LEGEND_FONT_SIZE = 16     
LEGEND_TITLE_SIZE = 16    

MARKER_SIZE = 8           
SPECIAL_MARKER_SIZE = 12  
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antibiticsmedianwithknockdown_plates_12april", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "14april_Antibiticsmedianwithknockdown_normalcode")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a",
    "Control Group": "#808080",  
    "Baseline (T0)": "#808080",  
    "unknown function": "#000000",
}

anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Contour Overlay for: {file_name}...")
    
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    
    is_ctrl = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    # Category setup
    df_plot['Display_Category'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3']).fillna("unknown function")
    df_plot.loc[(df_plot['Timepoint'] == 'T0') & (~is_ctrl), 'Display_Category'] = 'Baseline (T0)'
    df_plot.loc[is_ctrl, 'Display_Category'] = 'Control Group'

    # Legend Logic
    unique_in_data = df_plot['Display_Category'].unique()
    other_cats = sorted([c for c in unique_in_data if c not in MANUAL_COLORS])
    final_legend_order = ["Control Group", "Baseline (T0)"] + list(MANUAL_COLORS.keys())[0:13] + other_cats + ["unknown function"]
    
    # Updated Color Logic: Use #292727 for anything not in the manual list
    color_lookup = MANUAL_COLORS.copy()
    for cat in other_cats:
        color_lookup[cat] = "#292727"

    fig = go.Figure()

    # --- 3. KDE CONTOUR LOGIC ---
    mask_not_special = ~df_plot['Plate'].astype(str).str.contains('PLATE6|PLATE7')
    ctrl_df_for_kde = df_plot[(df_plot['Display_Category'] == 'Control Group') & mask_not_special]

    if not ctrl_df_for_kde.empty and len(ctrl_df_for_kde) > 5:
        x_pts, y_pts = ctrl_df_for_kde['UMAP1'].values, ctrl_df_for_kde['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        
        densities = kde(np.vstack([x_pts, y_pts]))
        levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
        
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=levels)
        for i, segs in enumerate(cs.allsegs):
            for j, seg in enumerate(segs):
                fig.add_trace(go.Scatter(
                    x=seg[:,0], y=seg[:,1], mode='lines',
                    line=dict(color='#808080', width=1.5),
                    showlegend=False, hoverinfo='skip'
                ))
        plt.close(fig_tmp)

    # --- 4. DATA POINTS (Layered by Z-Order) ---
    for is_special_pass in [False, True]:
        for cat in final_legend_order[::-1]:
            if cat not in df_plot['Display_Category'].values: continue
            
            sub_df = df_plot[df_plot['Display_Category'] == cat]
            
            plot_x, plot_y, plot_txt, symbols, colors, sizes = [], [], [], [], [], []
            
            for _, row in sub_df.iterrows():
                row_is_ctrl = str(row['Treatment']).lower() in ["no_sgrna", "nosgrna"]
                row_plate_str = str(row['Plate'])
                row_plate_special = "PLATE6" in row_plate_str or "PLATE7" in row_plate_str
                
                if row_plate_special != is_special_pass:
                    continue

                if row_is_ctrl and not row_plate_special:
                    continue 
                
                plot_x.append(row['UMAP1'])
                plot_y.append(row['UMAP2'])
                plot_txt.append(row['Treatment'])
                
                if row_plate_special:
                    sizes.append(SPECIAL_MARKER_SIZE)
                    if row_is_ctrl:
                        symbols.append("square")
                        # CHANGE: Specifically make Plate 6/7 controls black
                        colors.append("#000000")
                    else:
                        symbols.append("diamond")
                        colors.append(color_lookup.get(cat, "#292727"))
                else:
                    sizes.append(MARKER_SIZE)
                    symbols.append("circle")
                    colors.append(color_lookup.get(cat, "#292727"))

            if not plot_x: continue

            fig.add_trace(go.Scatter(
                x=plot_x, y=plot_y, mode='markers',
                name=cat, legendgroup=cat, showlegend=False,
                marker=dict(
                    size=sizes, 
                    color=colors,
                    symbol=symbols,
                    opacity=MARKER_OPACITY, 
                    line=dict(width=0.4, color='white')
                ),
                text=plot_txt,
                hovertemplate="<b>%{text}</b><br>Cat: " + cat + "<extra></extra>"
            ))

    # --- 5. FIXED SQUARE LAYOUT ---
    fig.update_layout(
        width=1400, height=900, autosize=False,
        template='plotly_white', font=dict(family="Arial"),
        legend=dict(
            itemsizing='constant', font=dict(size=LEGEND_FONT_SIZE),
            x=0.82, y=1, xanchor='left', yanchor='top'
        ),
        margin=dict(l=100, r=350, t=80, b=100),
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE), tickformat='.0f',
            constrain='domain', domain=[0, 0.75]
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
            showgrid=False, zeroline=False, showline=True, 
            linecolor='black', linewidth=AXIS_LINE_WIDTH, 
            ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
            tickfont=dict(size=AXIS_TICK_FONT_SIZE), tickformat='.0f',
            scaleanchor="x", scaleratio=1, constrain='domain'
        )
    )

    seen = set()
    for cat in final_legend_order:
        for trace in fig.data:
            if trace.name == cat and cat not in seen:
                trace.showlegend = True
                seen.add(cat)
                trace.legendrank = final_legend_order.index(cat)

    base_name = file_name.replace("Coords_", "UMAP_Contour_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".svg")))
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("\nDone. Controls on Plate 6/7 are now Black Squares; others remain unchanged.")

In [ ]:
#de vershcillende overlays bij antibiotics, in de pot antiobitcs met knockdown

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 4       
AXIS_TICK_WIDTH = 4       
AXIS_TICK_LEN = 10         
AXIS_TICK_FONT_SIZE = 30  
AXIS_TITLE_FONT_SIZE = 30

LEGEND_FONT_SIZE = 30     
LEGEND_TITLE_SIZE = 30    

MARKER_SIZE = 8           
SPECIAL_MARKER_SIZE = 12  
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  

# Colors for Hierarchy
COLOR_AXIS = '#000000'
COLOR_CONTOUR = "#565555"   
COLOR_OTHER_DOTS = "#787575" 

# Professional Color Palette (D3 Category 10)
SCIENTIFIC_PALETTE = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#bcbd22', '#17becf'
]

# --- 1. SETUP & SORTING LOGIC ---
PROJECT_ROOT = r'E:\Thesis3april'
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antibiticsmedianwithknockdown_plates_12april", "coordinates")
BASE_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "AntitbioticsmetknockdownOverlays_Metadata12april")

# Your specified order
SORT_ORDER = [
    'rifampicin', 'ciprofloxacin', 
    'cefotaxime', 'cefalexin', 'vancomycin', 
    'gentamycin', 'kanamycin', 'tetracycline'
]

moa_map = {
    'vancomycin': 'Cell Wall', 'cefalexin': 'Cell Wall', 'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA', 'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 'kanamycin': 'Ribosome', 'tetracycline': 'Ribosome',
    'LB (control)': 'Control'
}

OVERLAY_CATEGORIES = ['Treatment', 'Treatment_global', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']

for cat in OVERLAY_CATEGORIES:
    path = os.path.join(BASE_OUTPUT_DIR, cat)
    if not os.path.exists(path):
        os.makedirs(path)

def get_sort_key(val):
    """Helper to sort strings based on antibiotic names within them."""
    val_lower = str(val).lower()
    for i, drug in enumerate(SORT_ORDER):
        if drug in val_lower:
            return i
    return len(SORT_ORDER) 

# --- 2. METADATA PARSING FUNCTION ---
def parse_all_metadata(row):
    treat = str(row['Treatment'])
    plate_raw = str(row['Plate'])
    is_control = any(ctrl in treat.lower() for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        treat_global = "LB (control)"
        antibiotic = "LB (control)"
        moa = "Control"
        conc = "LB (control)"
        time = "LB (control)"
    else:
        treat_global = treat.split('_')[0]
        antibiotic = treat.split('_')[0]
        moa = moa_map.get(antibiotic, "Other")
        if "_10_" in treat or treat.endswith("_10"): conc = "10 MIC"
        elif "_50_" in treat or treat.endswith("_50"): conc = "50 MIC"
        else: conc = "Unknown"
        if treat.endswith('_1'): time = "T1"
        elif treat.endswith('_2'): time = "T2"
        else: time = "T0"
    
    clean_plate = "PLATE6" if "PLATE6" in plate_raw.upper() else ("PLATE7" if "PLATE7" in plate_raw.upper() else "Standard")
    return pd.Series([treat_global, antibiotic, moa, conc, time, clean_plate])

# --- 3. EXECUTION LOOP ---
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

for file_name in coord_files:
    print(f"Processing all overlays for: {file_name}...")
    df = pd.read_csv(os.path.join(COORD_DIR, file_name))
    df[['Treatment_global', 'Antibiotic', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']] = df.apply(parse_all_metadata, axis=1)
    
    is_special = df['Plate_Clean'].isin(["PLATE6", "PLATE7"])
    is_ctrl = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])

    for overlay_type in OVERLAY_CATEGORIES:
        fig = go.Figure()

        # A. KDE CONTOUR LOGIC (Original Method)
        ctrl_df_for_kde = df[is_ctrl & (~is_special)]
        if not ctrl_df_for_kde.empty and len(ctrl_df_for_kde) > 5:
            x_pts, y_pts = ctrl_df_for_kde['UMAP1'].values, ctrl_df_for_kde['UMAP2'].values
            kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
            densities = kde(np.vstack([x_pts, y_pts]))
            levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
            
            x_grid = np.linspace(df['UMAP1'].min()-2, df['UMAP1'].max()+2, 100)
            y_grid = np.linspace(df['UMAP2'].min()-2, df['UMAP2'].max()+2, 100)
            X, Y = np.meshgrid(x_grid, y_grid)
            Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

            fig_tmp, ax_tmp = plt.subplots()
            cs = ax_tmp.contour(X, Y, Z, levels=levels)
            for i, segs in enumerate(cs.allsegs):
                for j, seg in enumerate(segs):
                    fig.add_trace(go.Scatter(
                        x=seg[:,0], y=seg[:,1], mode='lines',
                        line=dict(color=COLOR_CONTOUR, width=1.5),
                        showlegend=False, hoverinfo='skip'
                    ))
            plt.close(fig_tmp)

        # B. BASE PASS: Other Plates (Updated to show Metadata + Well_ID)
        standard_mutants = df[~is_special & ~is_ctrl]
        if not standard_mutants.empty:
            fig.add_trace(go.Scatter(
                x=standard_mutants['UMAP1'], y=standard_mutants['UMAP2'],
                mode='markers', name='Other Plates',
                marker=dict(size=MARKER_SIZE, color=COLOR_OTHER_DOTS, opacity=0.4, line=dict(width=0.3, color='white')),
                showlegend=False,
                customdata=standard_mutants[['Well_ID', 'Plate']],
                text=standard_mutants['Treatment'],
                hovertemplate="<b>%{text}</b><br>Well: %{customdata[0]}<br>Plate: %{customdata[1]}<extra></extra>"
            ))

        # C. OVERLAY PASS: Plate 6 and 7
        special_df = df[is_special]
        unique_vals = [v for v in special_df[overlay_type].unique() if v != "LB (control)"]
        unique_vals.sort(key=get_sort_key) 
        color_map = {val: SCIENTIFIC_PALETTE[i % len(SCIENTIFIC_PALETTE)] for i, val in enumerate(unique_vals)}

        # 1. Mutants (Diamonds)
        for val in unique_vals:
            plot_df = special_df[(special_df[overlay_type] == val) & (~is_ctrl)]
            if plot_df.empty: continue
            fig.add_trace(go.Scatter(
                x=plot_df['UMAP1'], y=plot_df['UMAP2'],
                mode='markers', name=str(val),
                legendgroup=str(val),
                marker=dict(size=SPECIAL_MARKER_SIZE, color=color_map[val], symbol='diamond',
                            opacity=1.0, line=dict(width=0.5, color='white')),
                customdata=plot_df[['Well_ID']],
                text=plot_df['Treatment'],
                hovertemplate="<b>%{text}</b><br>Well: %{customdata[0]}<br>" + f"{overlay_type}: {val}<extra></extra>"
            ))

        # 2. Controls (Black Squares)
        ctrl_special = special_df[is_ctrl]
        if not ctrl_special.empty:
            fig.add_trace(go.Scatter(
                x=ctrl_special['UMAP1'], y=ctrl_special['UMAP2'],
                mode='markers', name='Control (P6/7)',
                marker=dict(size=SPECIAL_MARKER_SIZE, color='black', symbol='square',
                            opacity=1.0, line=dict(width=0.5, color='white')),
                customdata=ctrl_special[['Well_ID']],
                text=ctrl_special['Treatment'],
                hovertemplate="<b>%{text}</b><br>Well: %{customdata[0]}<br>Control Group<extra></extra>"
            ))

        # D. LAYOUT
        fig.update_layout(
            width=1400, height=900, autosize=False,
            template='plotly_white', font=dict(family="Arial"),
            legend=dict(itemsizing='constant', font=dict(size=LEGEND_FONT_SIZE),
                        x=0.82, y=1, xanchor='left', yanchor='top'),
            margin=dict(l=100, r=350, t=80, b=100),
            xaxis=dict(
                title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                showgrid=False, zeroline=False, showline=True, 
                linecolor=COLOR_AXIS, linewidth=AXIS_LINE_WIDTH, 
                ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                tickfont=dict(size=AXIS_TICK_FONT_SIZE), tickformat='.0f',
                constrain='domain', domain=[0, 0.75]
            ),
            yaxis=dict(
                title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                showgrid=False, zeroline=False, showline=True, 
                linecolor=COLOR_AXIS, linewidth=AXIS_LINE_WIDTH, 
                ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                tickfont=dict(size=AXIS_TICK_FONT_SIZE), tickformat='.0f',
                scaleanchor="x", scaleratio=1, constrain='domain'
            )
        )

        # --- SAVING OUTPUTS ---
        folder_path = os.path.join(BASE_OUTPUT_DIR, overlay_type)
        base_filename = f"Overlay_{overlay_type}_{file_name.replace('.csv', '')}"

        fig.write_image(os.path.join(folder_path, f"{base_filename}.png"), scale=PNG_RESOLUTION_SCALE)
        fig.write_image(os.path.join(folder_path, f"{base_filename}.svg"))
        fig.write_html(os.path.join(folder_path, f"{base_filename}.html"))

print(f"\nDone. PNG, SVG, and Interactive HTML plots generated in: {BASE_OUTPUT_DIR}")

In [ ]:
#antiibotica color op knockdown color 14 april: beetje chaos+assen enz nog niet juist

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 4       
AXIS_TICK_WIDTH = 4       
AXIS_TICK_LEN = 10         
AXIS_TICK_FONT_SIZE = 30  
AXIS_TITLE_FONT_SIZE = 30
LEGEND_FONT_SIZE = 16     
MARKER_SIZE = 8           
SPECIAL_MARKER_SIZE = 12  
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  

# Colors
COLOR_CONTOUR = "#808080"
COLOR_BASELINE = "#808080"
COLOR_UNANNOTATED = "#787575" # Muted grey for T1s without a manual pathway match

SCIENTIFIC_PALETTE = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#bcbd22', '#17becf'
]

MANUAL_COLORS = {
    "biosynthesis of peptidoglycan": "#1f77b4", 
    "biosynthesis of teichoic acid": "#ff7f0e", 
    "aminoacyl-tRNA synthetases": "#2ca02c",      
    "cell shape": "#d62728",                          
    "biosynthesis of fatty acids": "#9467bd",     
    "DNA replication": "#8c564b",                  
    "DNA condensation/ segregation": "#e377c2",   
    "cell division": "#bcbd22",                    
    "ribosomal proteins": "#17becf",              
    "biosynthesis of iron-sulfur clusters": "#aec7e8", 
    "glycolysis": "#ffbb78",                        
    "biosynthesis of menaquinone": "#98df8a",
}

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antibiticsmedianwithknockdown_plates_12april", "coordinates")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "14aprilcolorantibiotics__ColorPathway_Overlay")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

anno_df = pd.read_excel(anno_path)
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

# --- 2. EXECUTION LOOP ---
for file_name in coord_files:
    print(f"Generating Specialized Overlay for: {file_name}...")
    
    df_coords = pd.read_csv(os.path.join(COORD_DIR, file_name))
    df_plot = df_coords.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
    
    # Identify key groups
    df_plot['Timepoint'] = df_plot['Plate'].str.extract(r'_(T\d)')
    df_plot['Is_Special_Plate'] = df_plot['Plate'].str.contains('PLATE6|PLATE7')
    df_plot['Is_Control'] = df_plot['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
    
    # Determine Pathway category
    df_plot['Pathway'] = df_plot['SubtiWiki Annotation 4'].fillna(df_plot['SubtiWiki Annotation 3'])

    fig = go.Figure()

    # --- 3. KDE CONTOUR (Baseline Controls) ---
    ctrl_kde = df_plot[df_plot['Is_Control'] & ~df_plot['Is_Special_Plate']]
    if not ctrl_kde.empty and len(ctrl_kde) > 5:
        x_pts, y_pts = ctrl_kde['UMAP1'].values, ctrl_kde['UMAP2'].values
        kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
        x_grid = np.linspace(df_plot['UMAP1'].min()-2, df_plot['UMAP1'].max()+2, 100)
        y_grid = np.linspace(df_plot['UMAP2'].min()-2, df_plot['UMAP2'].max()+2, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
        
        fig_tmp, ax_tmp = plt.subplots()
        cs = ax_tmp.contour(X, Y, Z, levels=[np.percentile(kde(np.vstack([x_pts, y_pts])), p) for p in [5, 30, 55, 80]])
        for segs in cs.allsegs:
            for seg in segs:
                fig.add_trace(go.Scatter(x=seg[:,0], y=seg[:,1], mode='lines', 
                                         line=dict(color=COLOR_CONTOUR, width=1.5), showlegend=False, hoverinfo='skip'))
        plt.close(fig_tmp)

    # --- 4. DATA PLOTTING (Layered) ---
    
    # A. Baseline T0 (Grey Circles)
    t0_df = df_plot[(df_plot['Timepoint'] == 'T0') & (~df_plot['Is_Control'])]
    fig.add_trace(go.Scatter(x=t0_df['UMAP1'], y=t0_df['UMAP2'], mode='markers', name='Baseline (T0)',
                             marker=dict(size=MARKER_SIZE, color=COLOR_BASELINE, opacity=0.5, symbol='circle'),
                             text=t0_df['Treatment'], hovertemplate="<b>%{text}</b><br>Baseline T0<extra></extra>"))

    # B. T1 Knockdowns (Pathway Colored Circles)
    t1_df = df_plot[(df_plot['Timepoint'] == 'T1') & (~df_plot['Is_Special_Plate']) & (~df_plot['Is_Control'])]
    for path, color in MANUAL_COLORS.items():
        sub = t1_df[t1_df['Pathway'] == path]
        if not sub.empty:
            fig.add_trace(go.Scatter(x=sub['UMAP1'], y=sub['UMAP2'], mode='markers', name=path,
                                     marker=dict(size=MARKER_SIZE, color=color, opacity=MARKER_OPACITY),
                                     text=sub['Treatment']))

    # C. Antibiotic Mutants (Plate 6/7 - Scientific Palette Diamonds)
    special_mutants = df_plot[df_plot['Is_Special_Plate'] & ~df_plot['Is_Control']]
    if not special_mutants.empty:
        # Sort or group by treatment to apply the palette
        unique_treats = special_mutants['Treatment'].unique()
        for i, treat in enumerate(unique_treats):
            sub = special_mutants[special_mutants['Treatment'] == treat]
            fig.add_trace(go.Scatter(x=sub['UMAP1'], y=sub['UMAP2'], mode='markers', name=treat,
                                     marker=dict(size=SPECIAL_MARKER_SIZE, symbol='diamond', 
                                                 color=SCIENTIFIC_PALETTE[i % len(SCIENTIFIC_PALETTE)]),
                                     text=sub['Treatment']))

    # D. Plate 6/7 Controls (Black Squares)
    special_ctrls = df_plot[df_plot['Is_Special_Plate'] & df_plot['Is_Control']]
    if not special_ctrls.empty:
        fig.add_trace(go.Scatter(x=special_ctrls['UMAP1'], y=special_ctrls['UMAP2'], mode='markers', name='Control (P6/7)',
                                 marker=dict(size=SPECIAL_MARKER_SIZE, color='black', symbol='square'),
                                 text=special_ctrls['Treatment']))

    # --- 5. LAYOUT ---
    fig.update_layout(
        width=1400, height=900, template='plotly_white',
        legend=dict(font=dict(size=LEGEND_FONT_SIZE), x=0.82, y=1),
        margin=dict(l=100, r=350, t=80, b=100),
        xaxis=dict(title="UMAP 1", linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=True,
                   ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                   constrain='domain', domain=[0, 0.75]),
        yaxis=dict(title="UMAP 2", linecolor='black', linewidth=AXIS_LINE_WIDTH, mirror=True,
                   ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                   scaleanchor="x", scaleratio=1, constrain='domain')
    )

    base_name = file_name.replace("Coords_", "UMAP_Special_")
    fig.write_image(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".png")), scale=PNG_RESOLUTION_SCALE)
    fig.write_html(os.path.join(OUTPUT_DIR, base_name.replace(".csv", ".html")))

print("Done.")

In [ ]:
#antibiotica op knockdown plotten with cross targets visualized

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
import scipy.stats as st
import matplotlib.pyplot as plt

# --- 0. ADAPTABLE STYLE PARAMETERS ---
AXIS_LINE_WIDTH = 4       
AXIS_TICK_WIDTH = 4       
AXIS_TICK_LEN = 10         
AXIS_TICK_FONT_SIZE = 30  
AXIS_TITLE_FONT_SIZE = 30

LEGEND_FONT_SIZE = 30     
LEGEND_TITLE_SIZE = 30    

MARKER_SIZE = 8            
SPECIAL_MARKER_SIZE = 12  
HIGHLIGHT_CROSS_SIZE = 18  # Size for the highlighted gene crosses
MARKER_OPACITY = 0.8      
PNG_RESOLUTION_SCALE = 7  

# --- NEW HIGHLIGHT LIST ---
# Add the gene names you want to highlight here
HIGHLIGHT_GENES = ["rpoB", 
    "gyrA", 
    "gyrB", 
    "grlA", 
    "grlB", 
    "pbpB", 
    "ponA", 
    "pbpA", 
    "rrs"
]

# Colors for Hierarchy
COLOR_AXIS = '#000000'
COLOR_CONTOUR = "#565555"   
COLOR_OTHER_DOTS = "#787575" 

# Professional Color Palette (D3 Category 10)
SCIENTIFIC_PALETTE = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#bcbd22', '#17becf'
]

# --- 1. SETUP & SORTING LOGIC ---
PROJECT_ROOT = r'E:\Thesis3april'
COORD_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antibiticsmedianwithknockdown_plates_12april", "coordinates")
BASE_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "Antitbioticsmetknockdown_targetCross_Metadata12april")

SORT_ORDER = [
    'rifampicin', 'ciprofloxacin', 
    'cefotaxime', 'cefalexin', 'vancomycin', 
    'gentamycin', 'kanamycin', 'tetracycline'
]

moa_map = {
    'vancomycin': 'Cell Wall', 'cefalexin': 'Cell Wall', 'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA', 'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 'kanamycin': 'Ribosome', 'tetracycline': 'Ribosome',
    'LB (control)': 'Control'
}

OVERLAY_CATEGORIES = ['Treatment', 'Treatment_global', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']

for cat in OVERLAY_CATEGORIES:
    path = os.path.join(BASE_OUTPUT_DIR, cat)
    if not os.path.exists(path):
        os.makedirs(path)

def get_sort_key(val):
    val_lower = str(val).lower()
    for i, drug in enumerate(SORT_ORDER):
        if drug in val_lower:
            return i
    return len(SORT_ORDER) 

# --- 2. METADATA PARSING FUNCTION ---
def parse_all_metadata(row):
    treat = str(row['Treatment'])
    plate_raw = str(row['Plate'])
    is_control = any(ctrl in treat.lower() for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        treat_global = "LB (control)"
        antibiotic = "LB (control)"
        moa = "Control"
        conc = "LB (control)"
        time = "LB (control)"
    else:
        treat_global = treat.split('_')[0]
        antibiotic = treat.split('_')[0]
        moa = moa_map.get(antibiotic, "Other")
        if "_10_" in treat or treat.endswith("_10"): conc = "10 MIC"
        elif "_50_" in treat or treat.endswith("_50"): conc = "50 MIC"
        else: conc = "Unknown"
        if treat.endswith('_1'): time = "T1"
        elif treat.endswith('_2'): time = "T2"
        else: time = "T0"
    
    clean_plate = "PLATE6" if "PLATE6" in plate_raw.upper() else ("PLATE7" if "PLATE7" in plate_raw.upper() else "Standard")
    return pd.Series([treat_global, antibiotic, moa, conc, time, clean_plate])

# --- 3. EXECUTION LOOP ---
coord_files = [f for f in os.listdir(COORD_DIR) if f.endswith('.csv')]

for file_name in coord_files:
    print(f"Processing all overlays for: {file_name}...")
    df = pd.read_csv(os.path.join(COORD_DIR, file_name))
    df[['Treatment_global', 'Antibiotic', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']] = df.apply(parse_all_metadata, axis=1)
    
    is_special = df['Plate_Clean'].isin(["PLATE6", "PLATE7"])
    is_ctrl = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])

    for overlay_type in OVERLAY_CATEGORIES:
        fig = go.Figure()

        # A. KDE CONTOUR LOGIC
        ctrl_df_for_kde = df[is_ctrl & (~is_special)]
        if not ctrl_df_for_kde.empty and len(ctrl_df_for_kde) > 5:
            x_pts, y_pts = ctrl_df_for_kde['UMAP1'].values, ctrl_df_for_kde['UMAP2'].values
            kde = st.gaussian_kde(np.vstack([x_pts, y_pts]))
            densities = kde(np.vstack([x_pts, y_pts]))
            levels = [np.percentile(densities, p) for p in [5, 30, 55, 80]]
            
            x_grid = np.linspace(df['UMAP1'].min()-2, df['UMAP1'].max()+2, 100)
            y_grid = np.linspace(df['UMAP2'].min()-2, df['UMAP2'].max()+2, 100)
            X, Y = np.meshgrid(x_grid, y_grid)
            Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

            fig_tmp, ax_tmp = plt.subplots()
            cs = ax_tmp.contour(X, Y, Z, levels=levels)
            for i, segs in enumerate(cs.allsegs):
                for j, seg in enumerate(segs):
                    fig.add_trace(go.Scatter(
                        x=seg[:,0], y=seg[:,1], mode='lines',
                        line=dict(color=COLOR_CONTOUR, width=1.5),
                        showlegend=False, hoverinfo='skip'
                    ))
            plt.close(fig_tmp)

        # B. BASE PASS: Other Plates
        standard_mutants = df[~is_special & ~is_ctrl]
        if not standard_mutants.empty:
            fig.add_trace(go.Scatter(
                x=standard_mutants['UMAP1'], y=standard_mutants['UMAP2'],
                mode='markers', name='Other Plates',
                marker=dict(size=MARKER_SIZE, color=COLOR_OTHER_DOTS, opacity=0.4, line=dict(width=0.3, color='white')),
                showlegend=False,
                customdata=standard_mutants[['Well_ID', 'Plate']],
                text=standard_mutants['Treatment'],
                hovertemplate="<b>%{text}</b><br>Well: %{customdata[0]}<br>Plate: %{customdata[1]}<extra></extra>"
            ))

        # C. OVERLAY PASS: Plate 6 and 7
        special_df = df[is_special]
        unique_vals = [v for v in special_df[overlay_type].unique() if v != "LB (control)"]
        unique_vals.sort(key=get_sort_key) 
        color_map = {val: SCIENTIFIC_PALETTE[i % len(SCIENTIFIC_PALETTE)] for i, val in enumerate(unique_vals)}

        # 1. Mutants (Diamonds)
        for val in unique_vals:
            plot_df = special_df[(special_df[overlay_type] == val) & (~is_ctrl)]
            if plot_df.empty: continue
            fig.add_trace(go.Scatter(
                x=plot_df['UMAP1'], y=plot_df['UMAP2'],
                mode='markers', name=str(val),
                legendgroup=str(val),
                marker=dict(size=SPECIAL_MARKER_SIZE, color=color_map[val], symbol='diamond',
                            opacity=1.0, line=dict(width=0.5, color='white')),
                customdata=plot_df[['Well_ID']],
                text=plot_df['Treatment'],
                hovertemplate="<b>%{text}</b><br>Well: %{customdata[0]}<br>" + f"{overlay_type}: {val}<extra></extra>"
            ))

        # 2. Controls (Black Squares)
        ctrl_special = special_df[is_ctrl]
        if not ctrl_special.empty:
            fig.add_trace(go.Scatter(
                x=ctrl_special['UMAP1'], y=ctrl_special['UMAP2'],
                mode='markers', name='Control (P6/7)',
                marker=dict(size=SPECIAL_MARKER_SIZE, color='black', symbol='square',
                            opacity=1.0, line=dict(width=0.5, color='white')),
                customdata=ctrl_special[['Well_ID']],
                text=ctrl_special['Treatment'],
                hovertemplate="<b>%{text}</b><br>Well: %{customdata[0]}<br>Control Group<extra></extra>"
            ))

        # --- NEW: HIGHLIGHT PASS (Black Crosses) ---
        # Checks if any part of the treatment name contains the gene in the highlight list
        highlight_df = df[df['Treatment'].str.contains('|'.join(HIGHLIGHT_GENES), case=False, na=False)]
        
        if not highlight_df.empty:
            fig.add_trace(go.Scatter(
                x=highlight_df['UMAP1'], y=highlight_df['UMAP2'],
                mode='markers', 
                name='Highlighted Mutants',
                marker=dict(
                    size=HIGHLIGHT_CROSS_SIZE, 
                    color='black', 
                    symbol='x', # The cross shape
                    line=dict(width=2, color='black')
                ),
                text=highlight_df['Treatment'],
                hovertemplate="<b>Highlight: %{text}</b><extra></extra>",
                showlegend=True
            ))

        # D. LAYOUT
        fig.update_layout(
            width=1400, height=900, autosize=False,
            template='plotly_white', font=dict(family="Arial"),
            legend=dict(itemsizing='constant', font=dict(size=LEGEND_FONT_SIZE),
                        x=1.02, y=1, xanchor='left', yanchor='top'),
            margin=dict(l=100, r=400, t=80, b=100),
            xaxis=dict(
                title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                showgrid=False, zeroline=False, showline=True, 
                linecolor=COLOR_AXIS, linewidth=AXIS_LINE_WIDTH, 
                ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                tickfont=dict(size=AXIS_TICK_FONT_SIZE), tickformat='.0f',
                constrain='domain'
            ),
            yaxis=dict(
                title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                showgrid=False, zeroline=False, showline=True, 
                linecolor=COLOR_AXIS, linewidth=AXIS_LINE_WIDTH, 
                ticks="outside", tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN,
                tickfont=dict(size=AXIS_TICK_FONT_SIZE), tickformat='.0f',
                scaleanchor="x", scaleratio=1, constrain='domain'
            )
        )

        # --- SAVING OUTPUTS ---
        folder_path = os.path.join(BASE_OUTPUT_DIR, overlay_type)
        base_filename = f"Overlay_{overlay_type}_{file_name.replace('.csv', '')}"

        fig.write_image(os.path.join(folder_path, f"{base_filename}.png"), scale=PNG_RESOLUTION_SCALE)
        fig.write_image(os.path.join(folder_path, f"{base_filename}.svg"))
        fig.write_html(os.path.join(folder_path, f"{base_filename}.html"))

print(f"\nDone. PNG, SVG, and Interactive HTML plots generated in: {BASE_OUTPUT_DIR}")

In [ ]:
#antiiboitca apart plotten maar met features uti de selectie van hierboven = beste




#

In [ ]:
#bij files samen zetten staat er 191 common columns maar eiglk zijn er 187 kolomen in ifnale fiel dus ok

In [ ]:
import pandas as pd
import os

# Define the folder path
folder_path = r'E:\Thesis3april\files'

# Define the file names (adding .csv extension)
file1_name = 'antibiotics_aggregated_wells_median_min5_juist.csv'
file2_name = 'vettedcellcounts_median_T0_T1_met67_9april.csv'

# Create full paths
path1 = os.path.join(folder_path, file1_name)
path2 = os.path.join(folder_path, file2_name)

# 1. Load the datasets
df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)

# Identify non-feature columns (metadata)
# We want to keep these regardless, but use them to find the numerical features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']

# 2. Identify common feature columns
# This finds columns that exist in both DataFrames
common_cols = list(set(df1.columns).intersection(set(df2.columns)))

# Sort them to keep metadata first and features in order
# We prioritize keeping the metadata columns if they exist in the common list
ordered_cols = [c for c in metadata_cols if c in common_cols] + \
               sorted([c for c in common_cols if c not in metadata_cols], key=str)

# --- TASK 1: Filter the first file ---
df1_filtered = df1[ordered_cols]
output_path_filtered = os.path.join(folder_path, 'antibiotics_filtered_common_only_9apr.csv')
df1_filtered.to_csv(output_path_filtered, index=False)

# --- TASK 2: Merge both files ---
# We take the common columns from both and stack them
df2_filtered = df2[ordered_cols]
df_merged = pd.concat([df1_filtered, df2_filtered], ignore_index=True)

output_path_merged = os.path.join(folder_path, 'merged_datasets_common_features_9apr.csv')
df_merged.to_csv(output_path_merged, index=False)

print(f"Success!")
print(f"1. Filtered file saved to: {output_path_filtered}")
print(f"2. Merged file saved to: {output_path_merged}")
print(f"Number of common columns preserved: {len(ordered_cols)}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# ==========================================
# 0. ADAPTABLE STYLE PARAMETERS
# ==========================================
FONT_FAMILY = "Arial"
AXIS_LINE_WIDTH = 3
AXIS_TICK_WIDTH = 3
AXIS_TICK_LEN = 8
AXIS_TICK_FONT_SIZE = 24
AXIS_TITLE_FONT_SIZE = 24
LEGEND_FONT_SIZE = 24
MARKER_SIZE = 10 
MARKER_OPACITY = 0.85
PNG_RESOLUTION_SCALE = 2

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", 'antibiotics_filtered_common_only_9apr.csv')
df = pd.read_csv(file_path)
df.columns = [str(c) for c in df.columns]

# Main nested folder
BASE_OUTPUT = os.path.join(PROJECT_ROOT, "plots8april", "AntibioticsMedian_featureselectvanALLenP67_9april")
os.makedirs(BASE_OUTPUT, exist_ok=True)

# Define the MoA Mapping
moa_map = {
    'vancomycin': 'Cell Wall', 'cefalexin': 'Cell Wall', 'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA', 'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 'kanamycin': 'Ribosome', 'tetracycline': 'Ribosome',
    'LB (control)': 'Control'
}

# ==========================================
# 2. UNIFIED METADATA PARSING
# ==========================================
def parse_all_metadata(row):
    treat = str(row['Treatment'])
    plate_raw = str(row['Plate'])
    
    is_control = any(ctrl in treat.lower() for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        antibiotic = "LB (control)"
        moa = "Control"
        conc = "LB (control)"
        time = "LB (control)"
    else:
        antibiotic = treat.split('_')[0]
        moa = moa_map.get(antibiotic, "Other")
        conc = "10 MIC" if "10" in treat else ("50 MIC" if "50" in treat else "Unknown")
        if treat.endswith('_1'): time = "T1"
        elif treat.endswith('_2'): time = "T2"
        else: time = "T0"
    
    clean_plate = "PLATE6" if "PLATE6" in plate_raw.upper() else "PLATE7"
    
    return pd.Series([antibiotic, moa, conc, time, clean_plate])

df[['Antibiotic', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']] = df.apply(parse_all_metadata, axis=1)

# ==========================================
# 3. DEFINE CHANNEL SUBSETS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. MASTER PLOTTING FUNCTION
# ==========================================
def run_batch_umap(overlay_name, target_col, folder_name, palette):
    # This creates the subfolder WITHIN the Antibiotics folder
    output_dir = os.path.join(BASE_OUTPUT, folder_name)
    os.makedirs(output_dir, exist_ok=True)
    
    unique_items = sorted(df[target_col].unique())
    control_label = "LB (control)" if target_col != "MOA" else "Control"
    others = [i for i in unique_items if i != control_label]
    
    color_map = {item: palette[i % len(palette)] for i, item in enumerate(others)}
    color_map[control_label] = "#000000"

    for config in plot_configs:
        if not config['indices']: continue
        print(f"Processing {overlay_name}: {config['name']}...")
        
        X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
        reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42)
        embedding = reducer.fit_transform(X_scaled)
        
        df_plot = df.copy()
        df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
        
        fig = go.Figure()
        ordered_groups = [control_label] + sorted(others)

        for group in ordered_groups:
            sub_df = df_plot[df_plot[target_col] == group]
            if sub_df.empty: continue
            
            is_ctrl = (group == control_label)
            
            fig.add_trace(go.Scatter(
                x=sub_df['UMAP1'], y=sub_df['UMAP2'],
                mode='markers',
                name=group,
                marker=dict(
                    symbol='circle',
                    size=12 if is_ctrl else MARKER_SIZE,
                    color=color_map[group],
                    opacity=1.0 if is_ctrl else MARKER_OPACITY,
                    line=dict(width=0.8, color='white')
                ),
                text=sub_df['Treatment'],
                hovertemplate=f"<b>%{{text}}</b><br>{overlay_name}: {group}<extra></extra>"
            ))

        fig.update_layout(
            template='plotly_white', width=1350, height=900,
            margin=dict(l=100, r=50, b=100, t=80), 
            font=dict(family=FONT_FAMILY),
            legend=dict(
                font=dict(size=LEGEND_FONT_SIZE),
                title=dict(text=overlay_name, font=dict(size=LEGEND_FONT_SIZE)),
                x=0.65, y=1, xanchor='left', yanchor='top',
                itemsizing='constant', traceorder="normal"
            ),
            xaxis=dict(
                title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 0.6], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE)
            ),
            yaxis=dict(
                title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 1], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                scaleanchor="x", scaleratio=1
            )
        )

        base_fn = f"UMAP_{overlay_name}_{config['name']}"
        fig.write_image(os.path.join(output_dir, f"{base_fn}.png"), scale=PNG_RESOLUTION_SCALE)
        fig.write_image(os.path.join(output_dir, f"{base_fn}.svg"))
        fig.write_html(os.path.join(output_dir, f"{base_fn}.html"))

# ==========================================
# 5. RUN ALL BATCHES
# ==========================================
tasks = [
    ("Antibiotic", "Antibiotic", "Antibiotics_Mean", px.colors.qualitative.Dark24),
    ("MOA", "MOA", "Antibiotics_MOA", px.colors.qualitative.Bold),
    ("Concentration", "Concentration", "Antibiotics_Concentration", px.colors.qualitative.Set1),
    ("Timepoint", "Timepoint", "Antibiotics_Timepoints", px.colors.qualitative.Vivid),
    ("Plate", "Plate_Clean", "Antibiotics_Plates", px.colors.qualitative.Safe)
]

for name, col, folder, pal in tasks:
    run_batch_umap(name, col, folder, pal)

print(f"\nProcessing complete. All subfolders are located in: {BASE_OUTPUT}")

In [ ]:
#12april code same as above but with better colors

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# ==========================================
# 0. ADAPTABLE STYLE PARAMETERS
# ==========================================
FONT_FAMILY = "Arial"
AXIS_LINE_WIDTH = 4
AXIS_TICK_WIDTH = 4
AXIS_TICK_LEN = 10
AXIS_TICK_FONT_SIZE = 35
AXIS_TITLE_FONT_SIZE = 35
LEGEND_FONT_SIZE = 33
MARKER_SIZE = 16 
MARKER_OPACITY = 0.85
PNG_RESOLUTION_SCALE = 7

# Professional Color Palette (D3 Category 10)
SCIENTIFIC_PALETTE = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#bcbd22', '#17becf'
]

# Antibiotic Sort Order
SORT_ORDER = [
    'rifampicin', 'ciprofloxacin', 
    'cefotaxime', 'cefalexin', 'vancomycin', 
    'gentamycin', 'kanamycin', 'tetracycline'
]

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", 'antibiotics_filtered_common_only_9apr.csv')
df = pd.read_csv(file_path)
df.columns = [str(c) for c in df.columns]

BASE_OUTPUT = os.path.join(PROJECT_ROOT, "plots8april", "AntibioticsMedian_featureselectvanALLenP67_12april")
os.makedirs(BASE_OUTPUT, exist_ok=True)

moa_map = {
    'vancomycin': 'Cell Wall', 'cefalexin': 'Cell Wall', 'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA', 'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 'kanamycin': 'Ribosome', 'tetracycline': 'Ribosome',
    'LB (control)': 'Control'
}

def get_sort_key(val):
    val_lower = str(val).lower()
    for i, drug in enumerate(SORT_ORDER):
        if drug in val_lower:
            return i
    return len(SORT_ORDER)

# ==========================================
# 2. UNIFIED METADATA PARSING
# ==========================================
def parse_all_metadata(row):
    treat = str(row['Treatment'])
    plate_raw = str(row['Plate'])
    
    is_control = any(ctrl in treat.lower() for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        treat_global = "LB (control)"
        antibiotic = "LB (control)"
        moa = "Control"
        conc = "LB (control)"
        time = "LB (control)"
    else:
        treat_global = treat.split('_')[0]
        antibiotic = treat.split('_')[0]
        moa = moa_map.get(antibiotic, "Other")
        conc = "10 MIC" if "10" in treat else ("50 MIC" if "50" in treat else "Unknown")
        if treat.endswith('_1'): time = "T1"
        elif treat.endswith('_2'): time = "T2"
        else: time = "T0"
    
    clean_plate = "PLATE6" if "PLATE6" in plate_raw.upper() else "PLATE7"
    
    return pd.Series([treat_global, antibiotic, moa, conc, time, clean_plate])

df[['Treatment_global', 'Antibiotic', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']] = df.apply(parse_all_metadata, axis=1)

# ==========================================
# 3. DEFINE CHANNEL SUBSETS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. MASTER PLOTTING FUNCTION
# ==========================================
def run_batch_umap(overlay_name, target_col, folder_name):
    output_dir = os.path.join(BASE_OUTPUT, folder_name)
    os.makedirs(output_dir, exist_ok=True)
    
    unique_items = [i for i in df[target_col].unique() if i not in ["LB (control)", "Control"]]
    # Sort according to requested antibiotic order
    unique_items.sort(key=get_sort_key)
    
    control_label = "LB (control)" if target_col != "MOA" else "Control"
    
    # Apply the consistent Scientific Palette
    color_map = {item: SCIENTIFIC_PALETTE[i % len(SCIENTIFIC_PALETTE)] for i, item in enumerate(unique_items)}
    color_map[control_label] = "#000000"

    for config in plot_configs:
        if not config['indices']: continue
        print(f"Processing {overlay_name}: {config['name']}...")
        
        X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
        reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42)
        embedding = reducer.fit_transform(X_scaled)
        
        df_plot = df.copy()
        df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
        
        fig = go.Figure()
        ordered_groups = [control_label] + unique_items

        for group in ordered_groups:
            sub_df = df_plot[df_plot[target_col] == group]
            if sub_df.empty: continue
            
            is_ctrl = (group == control_label)
            
            fig.add_trace(go.Scatter(
                x=sub_df['UMAP1'], y=sub_df['UMAP2'],
                mode='markers',
                name=group,
                marker=dict(
                    symbol='circle',
                    size=16 if is_ctrl else MARKER_SIZE,
                    color=color_map[group],
                    opacity=1.0 if is_ctrl else MARKER_OPACITY,
                    line=dict(width=0.8, color='white')
                ),
                text=sub_df['Treatment'],
                hovertemplate=f"<b>%{{text}}</b><br>{overlay_name}: {group}<extra></extra>"
            ))

        fig.update_layout(
            template='plotly_white', width=1350, height=900,
            margin=dict(l=100, r=50, b=100, t=80), 
            font=dict(family=FONT_FAMILY),
            legend=dict(
                font=dict(size=LEGEND_FONT_SIZE),
                title=dict(text=overlay_name, font=dict(size=LEGEND_FONT_SIZE)),
                x=0.65, y=1, xanchor='left', yanchor='top',
                itemsizing='constant', traceorder="normal"
            ),
            xaxis=dict(
                title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 0.6], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE)
            ),
            yaxis=dict(
                title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 1], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                scaleanchor="x", scaleratio=1
            )
        )

        base_fn = f"UMAP_{overlay_name}_{config['name']}"
        fig.write_image(os.path.join(output_dir, f"{base_fn}.png"), scale=PNG_RESOLUTION_SCALE)
        fig.write_image(os.path.join(output_dir, f"{base_fn}.svg"))
        fig.write_html(os.path.join(output_dir, f"{base_fn}.html"))

# ==========================================
# 5. RUN ALL BATCHES
# ==========================================
tasks = [
    ("Antibiotic", "Antibiotic", "Antibiotics_Mean"),
    ("Treatment_global", "Treatment_global", "Antibiotics_Global"),
    ("MOA", "MOA", "Antibiotics_MOA"),
    ("Concentration", "Concentration", "Antibiotics_Concentration"),
    ("Timepoint", "Timepoint", "Antibiotics_Timepoints"),
    ("Plate", "Plate_Clean", "Antibiotics_Plates")
]

for name, col, folder in tasks:
    run_batch_umap(name, col, folder)

print(f"\nProcessing complete. All subfolders are located in: {BASE_OUTPUT}")

In [ ]:
#zelfde als hierboven maar andere neibours

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# ==========================================
# 0. ADAPTABLE STYLE PARAMETERS
# ==========================================
FONT_FAMILY = "Arial"
AXIS_LINE_WIDTH = 4
AXIS_TICK_WIDTH = 4
AXIS_TICK_LEN = 10
AXIS_TICK_FONT_SIZE = 30
AXIS_TITLE_FONT_SIZE = 30
LEGEND_FONT_SIZE = 30
MARKER_SIZE = 13 
MARKER_OPACITY = 0.85
PNG_RESOLUTION_SCALE = 7

# Professional Color Palette (D3 Category 10)
SCIENTIFIC_PALETTE = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#bcbd22', '#17becf'
]

# Antibiotic Sort Order
SORT_ORDER = [
    'rifampicin', 'ciprofloxacin', 
    'cefotaxime', 'cefalexin', 'vancomycin', 
    'gentamycin', 'kanamycin', 'tetracycline'
]

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", 'antibiotics_filtered_common_only_9apr.csv')
df = pd.read_csv(file_path)
df.columns = [str(c) for c in df.columns]

BASE_OUTPUT = os.path.join(PROJECT_ROOT, "plots8april", "AntibioticsMedian_featureselectvanALLenP67_anderneibhours_12april")
os.makedirs(BASE_OUTPUT, exist_ok=True)

moa_map = {
    'vancomycin': 'Cell Wall', 'cefalexin': 'Cell Wall', 'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA', 'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 'kanamycin': 'Ribosome', 'tetracycline': 'Ribosome',
    'LB (control)': 'Control'
}

def get_sort_key(val):
    val_lower = str(val).lower()
    for i, drug in enumerate(SORT_ORDER):
        if drug in val_lower:
            return i
    return len(SORT_ORDER)

# ==========================================
# 2. UNIFIED METADATA PARSING
# ==========================================
def parse_all_metadata(row):
    treat = str(row['Treatment'])
    plate_raw = str(row['Plate'])
    
    is_control = any(ctrl in treat.lower() for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        treat_global = "LB (control)"
        antibiotic = "LB (control)"
        moa = "Control"
        conc = "LB (control)"
        time = "LB (control)"
    else:
        treat_global = treat.split('_')[0]
        antibiotic = treat.split('_')[0]
        moa = moa_map.get(antibiotic, "Other")
        conc = "10 MIC" if "10" in treat else ("50 MIC" if "50" in treat else "Unknown")
        if treat.endswith('_1'): time = "T1"
        elif treat.endswith('_2'): time = "T2"
        else: time = "T0"
    
    clean_plate = "PLATE6" if "PLATE6" in plate_raw.upper() else "PLATE7"
    
    return pd.Series([treat_global, antibiotic, moa, conc, time, clean_plate])

df[['Treatment_global', 'Antibiotic', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']] = df.apply(parse_all_metadata, axis=1)

# ==========================================
# 3. DEFINE CHANNEL SUBSETS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. MASTER PLOTTING FUNCTION
# ==========================================
def run_batch_umap(overlay_name, target_col, folder_name):
    output_dir = os.path.join(BASE_OUTPUT, folder_name)
    os.makedirs(output_dir, exist_ok=True)
    
    unique_items = [i for i in df[target_col].unique() if i not in ["LB (control)", "Control"]]
    # Sort according to requested antibiotic order
    unique_items.sort(key=get_sort_key)
    
    control_label = "LB (control)" if target_col != "MOA" else "Control"
    
    # Apply the consistent Scientific Palette
    color_map = {item: SCIENTIFIC_PALETTE[i % len(SCIENTIFIC_PALETTE)] for i, item in enumerate(unique_items)}
    color_map[control_label] = "#000000"

    for config in plot_configs:
        if not config['indices']: continue
        print(f"Processing {overlay_name}: {config['name']}...")
        
        X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
        embedding = reducer.fit_transform(X_scaled)
        
        df_plot = df.copy()
        df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
        
        fig = go.Figure()
        ordered_groups = [control_label] + unique_items

        for group in ordered_groups:
            sub_df = df_plot[df_plot[target_col] == group]
            if sub_df.empty: continue
            
            is_ctrl = (group == control_label)
            
            fig.add_trace(go.Scatter(
                x=sub_df['UMAP1'], y=sub_df['UMAP2'],
                mode='markers',
                name=group,
                marker=dict(
                    symbol='circle',
                    size=12 if is_ctrl else MARKER_SIZE,
                    color=color_map[group],
                    opacity=1.0 if is_ctrl else MARKER_OPACITY,
                    line=dict(width=0.8, color='white')
                ),
                text=sub_df['Treatment'],
                hovertemplate=f"<b>%{{text}}</b><br>{overlay_name}: {group}<extra></extra>"
            ))

        fig.update_layout(
            template='plotly_white', width=1350, height=900,
            margin=dict(l=100, r=50, b=100, t=80), 
            font=dict(family=FONT_FAMILY),
            legend=dict(
                font=dict(size=LEGEND_FONT_SIZE),
                title=dict(text=overlay_name, font=dict(size=LEGEND_FONT_SIZE)),
                x=0.65, y=1, xanchor='left', yanchor='top',
                itemsizing='constant', traceorder="normal"
            ),
            xaxis=dict(
                title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 0.6], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE)
            ),
            yaxis=dict(
                title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 1], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                scaleanchor="x", scaleratio=1
            )
        )

        base_fn = f"UMAP_{overlay_name}_{config['name']}"
        fig.write_image(os.path.join(output_dir, f"{base_fn}.png"), scale=PNG_RESOLUTION_SCALE)
        fig.write_image(os.path.join(output_dir, f"{base_fn}.svg"))
        fig.write_html(os.path.join(output_dir, f"{base_fn}.html"))

# ==========================================
# 5. RUN ALL BATCHES
# ==========================================
tasks = [
    ("Antibiotic", "Antibiotic", "Antibiotics_Mean"),
    ("Treatment_global", "Treatment_global", "Antibiotics_Global"),
    ("MOA", "MOA", "Antibiotics_MOA"),
    ("Concentration", "Concentration", "Antibiotics_Concentration"),
    ("Timepoint", "Timepoint", "Antibiotics_Timepoints"),
    ("Plate", "Plate_Clean", "Antibiotics_Plates")
]

for name, col, folder in tasks:
    run_batch_umap(name, col, folder)

print(f"\nProcessing complete. All subfolders are located in: {BASE_OUTPUT}")

In [ ]:
#rommel









#rommel

In [ ]:
#zelfde als hierboven maar geen slechte controles weg: niet goed

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'

INPUT_CSV = os.path.join(PROJECT_ROOT, "files", "aggregated_wells_median_min5 _8april_metantibiotics.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT,"files")
CONTROL_LABEL = "no_sgRNA" 

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print("Loading raw data...")
df_raw_full = pd.read_csv(INPUT_CSV)
df_raw_full.columns = [str(c) for c in df_raw_full.columns]

# --- STEP: FILTER AND VIRTUAL MAPPING ---
df_raw = df_raw_full[df_raw_full['Plate'].str.contains('T0|T1')].copy()
df_raw['Virtual_TP'] = df_raw['Plate'].str[-2:]
mask = df_raw['Plate'].isin(['PLATE6_T1', 'PLATE7_T1'])
df_raw.loc[mask, 'Virtual_TP'] = 'T0'

metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count', 'Virtual_TP']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def get_channel_counts(features):
    """Counts features per channel based on numeric index ranges."""
    counts = {f"Channel_{i}": 0 for i in range(1, 6)}
    counts['Unknown/Out of Range'] = 0
    
    for f in features:
        try:
            val = int(f)
            if 0 <= val < 1280:
                counts["Channel_1"] += 1
            elif 1280 <= val < 2560:
                counts["Channel_2"] += 1
            elif 2560 <= val < 3840:
                counts["Channel_3"] += 1
            elif 3840 <= val < 5120:
                counts["Channel_4"] += 1
            elif 5120 <= val < 6400:
                counts["Channel_5"] += 1
            else:
                counts['Unknown/Out of Range'] += 1
        except ValueError:
            counts['Unknown/Out of Range'] += 1
    return counts

def filter_within_plate_consistency(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby(['Plate', 'Virtual_TP'])[features].median().reset_index()

    tp_batch_noises = []
    for tp in plate_medians['Virtual_TP'].unique():
        tp_subset = plate_medians[plate_medians['Virtual_TP'] == tp][features]
        if len(tp_subset) > 1:
            noise = tp_subset.std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features, len(to_drop)

# ==========================================
# EXECUTION PIPELINE
# ==========================================

print(f"\nStarting with {len(feature_cols)} total features.")

# --- STEP 0: GLOBAL VARIANCE ---
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
removed_s0 = len(feature_cols) - len(active_features)
print(f"Step 0 (Variance Filter): Removed {removed_s0}, {len(active_features)} remaining.")

# --- STEP 1: WITHIN-PLATE ---
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=2000)
removed_s1 = len(active_features) - len(step1_features)
print(f"Step 1 (Consistency):    Removed {removed_s1}, {len(step1_features)} remaining.")
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE ---
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
removed_s2 = len(step1_features) - len(step2_features)
print(f"Step 2 (Stability):      Removed {removed_s2}, {len(step2_features)} remaining.")
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY ---
final_feature_list, removed_s3 = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)
print(f"Step 3 (Redundancy):     Removed {removed_s3}, {len(final_feature_list)} remaining.")

# ==========================================
# FINAL REPORT & SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list].drop(columns=['Virtual_TP'])
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts_median_T0_T1_metAntibiotics_8april.csv")
df_final.to_csv(output_path, index=False)

channel_breakdown = get_channel_counts(final_feature_list)

print("\n" + "="*40)
print(f"FINAL FEATURE BREAKDOWN PER CHANNEL")
print("="*40)
for ch, count in channel_breakdown.items():
    if count > 0 or "Channel" in ch:
        print(f"{ch:10} : {count} features")

print(f"\nSaved results to: {output_path}")
print("="*40)

In [ ]:
#plotting antiobitc on knocdkwon but with old layout

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---



PROJECT_ROOT = r'E:\Thesis3april'




file_path = os.path.join(PROJECT_ROOT, "files", "vettedcellcounts_median_T0_T1_metAntibiotics_8april.csv")
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "plots8april", "vettedcellcounts_antibi_T0_T1_metAntibioticsall_8april")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading data...")
df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

# --- 2. GLOBAL VARIANCE FILTER ---
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c.isdigit()]
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
df = df_raw[metadata_cols + active_features].copy()

# --- 3. MERGE & CATEGORIZATION ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Identify groups
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
is_special_plate = df['Plate'].str.contains('PLATE6|PLATE7', case=False, na=False)

# NEW: Create a simplified Treatment name (everything before the first underscore)
# e.g., "vancomycin_50" becomes "vancomycin"
df['Treatment_Base'] = df['Treatment'].str.split('_').str[0]

# CATEGORIZATION LOGIC
df['Display_Category'] = df['Effective_Annotation']

# If it's PLATE6/7 and NOT a control, use the BASE Treatment name for coloring
df.loc[is_special_plate & ~is_control, 'Display_Category'] = df['Treatment_Base']

# Standard overrides
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control) & (~is_special_plate), 'Display_Category'] = 'Baseline (T0)'

# --- 4. GOLDEN ANGLE COLOR MAPPING ---
exclude = ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in exclude])
num_cats = len(all_cats)

if num_cats > 0:
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

color_map['no_sgrna'] = '#EBEBEB'
color_map['Baseline (T0)'] = '#B0B0B0'
color_map['Unknown/Other'] = '#222222'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Define shapes
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'
    df.loc[is_special_plate, 'Point_Shape'] = 'diamond'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square", "diamond": "diamond"},
        hover_name='Treatment', # Keep full name in hover
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # Trace/Legend clean up
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Styling
    fig.update_traces(marker=dict(opacity=0.9, line=dict(width=0.5, color='white'))) 
    
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=10, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square'))
    
    # Apply Large Diamond styling (colored by Treatment_Base)
    fig.update_traces(
        marker=dict(size=16, line=dict(width=1.5, color='black')), 
        selector=dict(marker_symbol='diamond')
    )
    
    fig.update_layout(
        width=1400, height=900,
        legend_title_text='Group / Compound (Diamonds)',
        xaxis=dict(title="UMAP 1", showline=True, linewidth=2, linecolor='black', showgrid=False),
        yaxis=dict(title="UMAP 2", showline=True, linewidth=2, linecolor='black', showgrid=False)
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))

print(f"Done. Plate 6/7 points are diamonds colored by compound name.")

In [ ]:
#other visualizaiton of antibiotics with knockdown

In [ ]:
#antibiotics plot but with features based on featuer sleciton of everything + antibiotics

In [ ]:
import pandas as pd
import os

# Define the folder path
folder_path = r'E:\Thesis3april\files'

# Define the file names (adding .csv extension)
file1_name = 'antibiotics_aggregated_wells_median_8april.csv'
file2_name = "vettedcellcounts_median_T0_T1_metAntibiotics_8april.csv"

# Create full paths
path1 = os.path.join(folder_path, file1_name)
path2 = os.path.join(folder_path, file2_name)

# 1. Load the datasets
df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)

# Identify non-feature columns (metadata)
# We want to keep these regardless, but use them to find the numerical features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']

# 2. Identify common feature columns
# This finds columns that exist in both DataFrames
common_cols = list(set(df1.columns).intersection(set(df2.columns)))

# Sort them to keep metadata first and features in order
# We prioritize keeping the metadata columns if they exist in the common list
ordered_cols = [c for c in metadata_cols if c in common_cols] + \
               sorted([c for c in common_cols if c not in metadata_cols], key=str)

# --- TASK 1: Filter the first file ---
df1_filtered = df1[ordered_cols]
output_path_filtered = os.path.join(folder_path, 'antibiotics_filtered_common_only_8april.csv')
df1_filtered.to_csv(output_path_filtered, index=False)

# --- TASK 2: Merge both files ---
# We take the common columns from both and stack them
df2_filtered = df2[ordered_cols]
df_merged = pd.concat([df1_filtered, df2_filtered], ignore_index=True)

output_path_merged = os.path.join(folder_path, 'merged_datasets_common_features_8april.csv')
df_merged.to_csv(output_path_merged, index=False)

print(f"Success!")
print(f"1. Filtered file saved to: {output_path_filtered}")
print(f"2. Merged file saved to: {output_path_merged}")
print(f"Number of common columns preserved: {len(ordered_cols)}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# ==========================================
# 0. ADAPTABLE STYLE PARAMETERS
# ==========================================
FONT_FAMILY = "Arial"
AXIS_LINE_WIDTH = 3
AXIS_TICK_WIDTH = 3
AXIS_TICK_LEN = 8
AXIS_TICK_FONT_SIZE = 24
AXIS_TITLE_FONT_SIZE = 24
LEGEND_FONT_SIZE = 24
MARKER_SIZE = 10 
MARKER_OPACITY = 0.85
PNG_RESOLUTION_SCALE = 2

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
PROJECT_ROOT = r'E:\Thesis3april'
file_path = os.path.join(PROJECT_ROOT, "files", 'antibiotics_filtered_common_only_8april.csv')
df = pd.read_csv(file_path)
df.columns = [str(c) for c in df.columns]

# Main nested folder
BASE_OUTPUT = os.path.join(PROJECT_ROOT, "plots3", "AntibioticsMedian_featureselectvanALLenP67_8april")
os.makedirs(BASE_OUTPUT, exist_ok=True)

# Define the MoA Mapping
moa_map = {
    'vancomycin': 'Cell Wall', 'cefalexin': 'Cell Wall', 'cefotaxime': 'Cell Wall',
    'rifampicin': 'RNA', 'ciprofloxacin': 'DNA',
    'gentamycin': 'Ribosome', 'kanamycin': 'Ribosome', 'tetracycline': 'Ribosome',
    'LB (control)': 'Control'
}

# ==========================================
# 2. UNIFIED METADATA PARSING
# ==========================================
def parse_all_metadata(row):
    treat = str(row['Treatment'])
    plate_raw = str(row['Plate'])
    
    is_control = any(ctrl in treat.lower() for ctrl in ["no_sgrna", "nosgrna"])
    
    if is_control:
        antibiotic = "LB (control)"
        moa = "Control"
        conc = "LB (control)"
        time = "LB (control)"
    else:
        antibiotic = treat.split('_')[0]
        moa = moa_map.get(antibiotic, "Other")
        conc = "10 MIC" if "10" in treat else ("50 MIC" if "50" in treat else "Unknown")
        if treat.endswith('_1'): time = "T1"
        elif treat.endswith('_2'): time = "T2"
        else: time = "T0"
    
    clean_plate = "PLATE6" if "PLATE6" in plate_raw.upper() else "PLATE7"
    
    return pd.Series([antibiotic, moa, conc, time, clean_plate])

df[['Antibiotic', 'MOA', 'Concentration', 'Timepoint', 'Plate_Clean']] = df.apply(parse_all_metadata, axis=1)

# ==========================================
# 3. DEFINE CHANNEL SUBSETS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. MASTER PLOTTING FUNCTION
# ==========================================
def run_batch_umap(overlay_name, target_col, folder_name, palette):
    # This creates the subfolder WITHIN the Antibiotics folder
    output_dir = os.path.join(BASE_OUTPUT, folder_name)
    os.makedirs(output_dir, exist_ok=True)
    
    unique_items = sorted(df[target_col].unique())
    control_label = "LB (control)" if target_col != "MOA" else "Control"
    others = [i for i in unique_items if i != control_label]
    
    color_map = {item: palette[i % len(palette)] for i, item in enumerate(others)}
    color_map[control_label] = "#000000"

    for config in plot_configs:
        if not config['indices']: continue
        print(f"Processing {overlay_name}: {config['name']}...")
        
        X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
        reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=42)
        embedding = reducer.fit_transform(X_scaled)
        
        df_plot = df.copy()
        df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
        
        fig = go.Figure()
        ordered_groups = [control_label] + sorted(others)

        for group in ordered_groups:
            sub_df = df_plot[df_plot[target_col] == group]
            if sub_df.empty: continue
            
            is_ctrl = (group == control_label)
            
            fig.add_trace(go.Scatter(
                x=sub_df['UMAP1'], y=sub_df['UMAP2'],
                mode='markers',
                name=group,
                marker=dict(
                    symbol='circle',
                    size=12 if is_ctrl else MARKER_SIZE,
                    color=color_map[group],
                    opacity=1.0 if is_ctrl else MARKER_OPACITY,
                    line=dict(width=0.8, color='white')
                ),
                text=sub_df['Treatment'],
                hovertemplate=f"<b>%{{text}}</b><br>{overlay_name}: {group}<extra></extra>"
            ))

        fig.update_layout(
            template='plotly_white', width=1350, height=900,
            margin=dict(l=100, r=50, b=100, t=80), 
            font=dict(family=FONT_FAMILY),
            legend=dict(
                font=dict(size=LEGEND_FONT_SIZE),
                title=dict(text=overlay_name, font=dict(size=LEGEND_FONT_SIZE)),
                x=0.65, y=1, xanchor='left', yanchor='top',
                itemsizing='constant', traceorder="normal"
            ),
            xaxis=dict(
                title=dict(text="UMAP 1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 0.6], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE)
            ),
            yaxis=dict(
                title=dict(text="UMAP 2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                domain=[0, 1], showline=True, linecolor='black', linewidth=AXIS_LINE_WIDTH, 
                showgrid=False, zeroline=False, ticks="outside",
                tickwidth=AXIS_TICK_WIDTH, ticklen=AXIS_TICK_LEN, tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                scaleanchor="x", scaleratio=1
            )
        )

        base_fn = f"UMAP_{overlay_name}_{config['name']}"
        fig.write_image(os.path.join(output_dir, f"{base_fn}.png"), scale=PNG_RESOLUTION_SCALE)
        fig.write_image(os.path.join(output_dir, f"{base_fn}.svg"))
        fig.write_html(os.path.join(output_dir, f"{base_fn}.html"))

# ==========================================
# 5. RUN ALL BATCHES
# ==========================================
tasks = [
    ("Antibiotic", "Antibiotic", "Antibiotics_Mean", px.colors.qualitative.Dark24),
    ("MOA", "MOA", "Antibiotics_MOA", px.colors.qualitative.Bold),
    ("Concentration", "Concentration", "Antibiotics_Concentration", px.colors.qualitative.Set1),
    ("Timepoint", "Timepoint", "Antibiotics_Timepoints", px.colors.qualitative.Vivid),
    ("Plate", "Plate_Clean", "Antibiotics_Plates", px.colors.qualitative.Safe)
]

for name, col, folder, pal in tasks:
    run_batch_umap(name, col, folder, pal)

print(f"\nProcessing complete. All subfolders are located in: {BASE_OUTPUT}")

In [ ]:
#####







statistics 






########

In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

# --- 1. SETUP & DATA CLEANING ---
FILE_PATH = r'E:\Thesis3april\overlay\all_individual_cells_areas.csv'
output_dir = r'E:\Thesis3april\statistics'
output_csv = os.path.join(output_dir, 'control_area_statistics_summary.csv')
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(FILE_PATH)

# Filter for controls
df_ctrl = df[df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])].copy()

# Extract Timepoint (T0, T1, T2) from Plate name
def extract_tp(plate_name):
    for tp in ['T0', 'T1', 'T2']:
        if tp in plate_name: return tp
    return None

df_ctrl['Timepoint'] = df_ctrl['Plate'].apply(extract_tp)
df_ctrl = df_ctrl.dropna(subset=['Timepoint'])

# --- 2. CALCULATE DESCRIPTIVE STATISTICS ---
def calculate_mad(series):
    return np.median(np.abs(series - series.median()))

summary = df_ctrl.groupby('Timepoint')['Area'].agg([
    ('Mean_Area', 'mean'),
    ('Median_Area', 'median'),
    ('Std_Dev', 'std'),
    ('Count', 'count')
]).reset_index()

mads = df_ctrl.groupby('Timepoint')['Area'].apply(calculate_mad).reset_index()
mads.columns = ['Timepoint', 'MAD']
summary = pd.merge(summary, mads, on='Timepoint')

# --- 3. CALCULATE P-VALUES ---
# Grab data for comparisons
t0_data = df_ctrl[df_ctrl['Timepoint'] == 'T0']['Area']
t1_data = df_ctrl[df_ctrl['Timepoint'] == 'T1']['Area']
t2_data = df_ctrl[df_ctrl['Timepoint'] == 'T2']['Area']

p_vs_t0 = {}
p_t2_vs_t1 = np.nan

for tp in summary['Timepoint'].unique():
    current_data = df_ctrl[df_ctrl['Timepoint'] == tp]['Area']
    
    # Comparison vs T0
    if tp == 'T0':
        p_vs_t0[tp] = np.nan
    else:
        _, p = mannwhitneyu(t0_data, current_data, alternative='two-sided')
        p_vs_t0[tp] = p
    
    # Specific T2 vs T1 Comparison
    if tp == 'T2' and not t1_data.empty:
        _, p_val = mannwhitneyu(t1_data, t2_data, alternative='two-sided')
        p_t2_vs_t1 = p_val

summary['p_value_vs_T0'] = summary['Timepoint'].map(p_vs_t0)
summary.loc[summary['Timepoint'] == 'T2', 'p_value_T2_vs_T1'] = p_t2_vs_t1

# --- 4. FORMATTING FOR OUTPUT ---
def format_p(p):
    if pd.isna(p): return "N/A"
    if p == 0 or p < 1e-300: return "< 1e-300"
    if p < 0.001: return f"{p:.2e}"
    return f"{p:.4f}"

summary['p_vs_T0_formatted'] = summary['p_value_vs_T0'].apply(format_p)
summary['p_T2vsT1_formatted'] = summary['p_value_T2_vs_T1'].apply(format_p)

# --- 5. PLOTTING DISTRIBUTIONS ---
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(18, 8)) 

tp_order = sorted(df_ctrl['Timepoint'].unique())

# Plot A: KDE Distribution
kde_plot = sns.kdeplot(data=df_ctrl, x='Area', hue='Timepoint', hue_order=tp_order, 
            fill=True, palette='viridis', common_norm=False, ax=axes[0])
axes[0].set_title('Cell Area Density Distribution', fontsize=16)
axes[0].set_xlim(0, df_ctrl['Area'].quantile(0.99))
axes[0].set_box_aspect(1) # Square plotted region

# Move legend for KDE plot to the right
sns.move_legend(axes[0], "upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0)

# Plot B: Boxplot
sns.boxplot(data=df_ctrl, x='Timepoint', y='Area', order=tp_order, 
            palette='viridis', ax=axes[1], showfliers=False)
axes[1].set_title('Area Spread (Outliers Hidden)', fontsize=16)
axes[1].set_box_aspect(1) # Square plotted region

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'control_area_distributions_T2vsT1.png'), dpi=300)
plt.show()

# --- 6. SAVE OUTPUT ---
summary.to_csv(output_csv, index=False)

print("--- Statistics Summary ---")
print(summary[['Timepoint', 'Median_Area', 'p_vs_T0_formatted', 'p_T2vsT1_formatted']])

In [ ]:
#right statistics method

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import stats

# Paths based on your setup
FILE_PATH = r'E:\Thesis3april\overlay\all_individual_cells_areas.csv'
output_dir = r'E:\Thesis3april\statistics'
os.makedirs(output_dir, exist_ok=True)

# Load and Filter
df = pd.read_csv(FILE_PATH)
df_ctrl = df[df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])].copy()

def extract_tp(plate_name):
    for tp in ['T0', 'T1', 'T2']:
        if tp in plate_name: return tp
    return None

df_ctrl['Timepoint'] = df_ctrl['Plate'].apply(extract_tp)
df_ctrl = df_ctrl.dropna(subset=['Timepoint'])

In [ ]:
print("--- Running Normality Checks ---")
normality_results = []
timepoints = sorted(df_ctrl['Timepoint'].unique())

for tp in timepoints:
    data = df_ctrl[df_ctrl['Timepoint'] == tp]['Area']
    
    # 1. Shapiro-Wilk Test
    # Note: If your N is very large (>5000), Shapiro is extremely sensitive.
    shapiro_stat, shapiro_p = stats.shapiro(data)
    
    normality_results.append({
        'Timepoint': tp,
        'Shapiro_Stat': shapiro_stat,
        'p_value': shapiro_p,
        'Is_Normal': shapiro_p > 0.05
    })
    
    # 2. Q-Q Plot (Saved as a square)
    plt.figure(figsize=(6, 6))
    stats.probplot(data, dist="norm", plot=plt)
    plt.title(f'Normality Q-Q Plot: {tp}')
    
    # Ensure square plotting region
    plt.gca().set_box_aspect(1) 
    
    plt.savefig(os.path.join(output_dir, f'normality_qq_{tp}.png'), bbox_inches='tight')
    plt.close()

# Save text report
normality_df = pd.DataFrame(normality_results)
normality_df.to_csv(os.path.join(output_dir, 'normality_report.csv'), index=False)

print(normality_df)

In [ ]:
!pip install scikit_posthocs

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp
import os

# --- 1. DEFINE DATA GROUPS ---
# We must re-define these within the cell to ensure the script finds them
t0_data = df_ctrl[df_ctrl['Timepoint'] == 'T0']['Area']
t1_data = df_ctrl[df_ctrl['Timepoint'] == 'T1']['Area']
t2_data = df_ctrl[df_ctrl['Timepoint'] == 'T2']['Area']

# --- 2. GLOBAL TEST (Kruskal-Wallis) ---
# Checks if there is ANY difference between the three groups
stat, p_global = stats.kruskal(t0_data, t1_data, t2_data)

print(f"Global Kruskal-Wallis p-value: {p_global:.4e}")

# --- 3. PAIRWISE COMPARISONS (Dunn's Test) ---
if p_global < 0.05:
    print("Significant difference found. Running Dunn's post-hoc test...")
    
    # We use the 'bonferroni' adjustment to correct for multiple comparisons
    # This gives us the specific T0 vs T1, T0 vs T2, and T1 vs T2 p-values
    posthoc_df = sp.posthoc_dunn(df_ctrl, val_col='Area', group_col='Timepoint', p_adjust='bonferroni')
    
    # Save the results to your statistics folder
    posthoc_csv = os.path.join(output_dir, 'pairwise_comparisons_dunn.csv')
    posthoc_df.to_csv(posthoc_csv)
    
    print("\n--- Pairwise P-Value Matrix (Bonferroni Corrected) ---")
    print(posthoc_df)
else:
    print("No significant difference found between groups (p > 0.05).")

In [ ]:
import scipy.stats as stats
import scikit_posthocs as sp
import os

# --- 1. DEFINE DATA GROUPS ---
t0_data = df_ctrl[df_ctrl['Timepoint'] == 'T0']['Area']
t1_data = df_ctrl[df_ctrl['Timepoint'] == 'T1']['Area']
t2_data = df_ctrl[df_ctrl['Timepoint'] == 'T2']['Area']

# --- 2. GLOBAL TEST (Kruskal-Wallis) ---
stat, p_global = stats.kruskal(t0_data, t1_data, t2_data)

print(f"Global Kruskal-Wallis p-value: {p_global:.4e}")

# --- 3. PAIRWISE COMPARISONS (Dunn's Test) ---
if p_global < 0.05:
    print("Significant difference found. Running Dunn's post-hoc test...")
    
    # Perform Dunn's test with Bonferroni correction
    posthoc_df = sp.posthoc_dunn(df_ctrl, val_col='Area', group_col='Timepoint',p_adjust='bonferroni') # Change from 'bonferroni' to 'fdr_bh'

    
    # Save the matrix
    posthoc_csv = os.path.join(output_dir, 'pairwise_comparisons_dunn.csv')
    posthoc_df.to_csv(posthoc_csv)
    
    print("\n--- Pairwise P-Value Matrix ---")
    print(posthoc_df)
    
    # --- 4. PRINT EXACT P-VALUES FOR YOUR THESIS ---
    print("\n--- Exact Pairwise P-Values (Scientific Notation) ---")
    # Extracting exact values from the matrix
    try:
        p_t0_t1 = posthoc_df.loc['T0', 'T1']
        p_t0_t2 = posthoc_df.loc['T0', 'T2']
        p_t1_t2 = posthoc_df.loc['T1', 'T2']

        print(f"T0 vs T1: p = {p_t0_t1:.4e}")
        print(f"T0 vs T2: p = {p_t0_t2:.4e}")
        print(f"T1 vs T2: p = {p_t1_t2:.4e}")
    except KeyError:
        print("Check your Timepoint labels (T0, T1, T2) to ensure they match the dataframe exactly.")

else:
    print("No significant difference found between groups (p > 0.05).")

In [ ]:
#statistics on clusters

In [ ]:
#fatty acid

In [ ]:
import pandas as pd
import os

# --- 1. INPUTS AND SETUP ---
# Update these paths to match your local setup
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "statistics", "evaluation_results")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Load your annotation data
anno_df = pd.read_excel(anno_path)

# --- 2. DEFINE YOUR DETECTIONS ---
# These are the genes you identified from the plot for the specific category
target_category = "biosynthesis of fatty acids"
my_detections = {'accA', 'accB', 'accC', 'accD', 'fabD', 'fabF', 'fabG', 'plsC', 'plsX'}

# --- 3. CALCULATION LOGIC ---
# Get Ground Truth from the Excel file based on your plotting logic
# We look at both Annotation 4 and 3
ground_truth_mask = (
    (anno_df['SubtiWiki Annotation 4'] == target_category) | 
    ((anno_df['SubtiWiki Annotation 4'].isna()) & (anno_df['SubtiWiki Annotation 3'] == target_category))
)
ground_truth_genes = set(anno_df.loc[ground_truth_mask, 'Treatment'].unique())
all_genes_in_dataset = set(anno_df['Treatment'].unique())

# Core Metrics
tp_genes = my_detections.intersection(ground_truth_genes)
fp_genes = my_detections - ground_truth_genes
fn_genes = ground_truth_genes - my_detections

tp = len(tp_genes)
fp = len(fp_genes)
fn = len(fn_genes)
tn = len(all_genes_in_dataset - (ground_truth_genes | my_detections))

# Math for Metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
accuracy = (tp + tn) / (tp + fp + fn + tn)
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# --- 4. CREATE SUMMARY AND SAVE ---
summary_data = {
    "Metric": ["Category", "True Positives", "False Positives", "False Negatives", "True Negatives", "Precision", "Recall", "Accuracy", "F1-Score"],
    "Value": [target_category, tp, fp, fn, tn, f"{precision:.4f}", f"{recall:.4f}", f"{accuracy:.4f}", f"{f1:.4f}"],
    "Genes_Involved": ["-", ", ".join(tp_genes), ", ".join(fp_genes), ", ".join(fn_genes), "Remaining dataset", "-", "-", "-", "-"]
}

summary_df = pd.DataFrame(summary_data)
csv_output_path = os.path.join(OUTPUT_DIR, "FattyAcidgene_detection_summary.csv")
summary_df.to_csv(csv_output_path, index=False)

print(f"Evaluation complete.")
print(f"Summary saved to: {csv_output_path}")

# Quick print to console for immediate checking
print("\n--- Quick Stats ---")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")

In [ ]:
#ribosomal

In [ ]:
import pandas as pd
import os

# --- 1. INPUTS AND SETUP ---
# Update these paths to match your local setup
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "statistics", "evaluation_results")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Load your annotation data
anno_df = pd.read_excel(anno_path)

# --- 2. DEFINE YOUR DETECTIONS ---
# These are the genes you identified from the plot for the specific category
target_category = "ribosomal proteins"
my_detections = {"rpsQ", "secY", "rpsNA", "rplC", "rpsJ", "rpsE", 
    "rplF", "rpsH", "rpsI", "rplM", "rplX", "rplR105", 
    "rpoC", "rpsK", "rplE", "sigA", "rpsR", "dnaG"}

# --- 3. CALCULATION LOGIC ---
# Get Ground Truth from the Excel file based on your plotting logic
# We look at both Annotation 4 and 3
ground_truth_mask = (
    (anno_df['SubtiWiki Annotation 4'] == target_category) | 
    ((anno_df['SubtiWiki Annotation 4'].isna()) & (anno_df['SubtiWiki Annotation 3'] == target_category))
)
ground_truth_genes = set(anno_df.loc[ground_truth_mask, 'Treatment'].unique())
all_genes_in_dataset = set(anno_df['Treatment'].unique())

# Core Metrics
tp_genes = my_detections.intersection(ground_truth_genes)
fp_genes = my_detections - ground_truth_genes
fn_genes = ground_truth_genes - my_detections

tp = len(tp_genes)
fp = len(fp_genes)
fn = len(fn_genes)
tn = len(all_genes_in_dataset - (ground_truth_genes | my_detections))

# Math for Metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
accuracy = (tp + tn) / (tp + fp + fn + tn)
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# --- 4. CREATE SUMMARY AND SAVE ---
summary_data = {
    "Metric": ["Category", "True Positives", "False Positives", "False Negatives", "True Negatives", "Precision", "Recall", "Accuracy", "F1-Score"],
    "Value": [target_category, tp, fp, fn, tn, f"{precision:.4f}", f"{recall:.4f}", f"{accuracy:.4f}", f"{f1:.4f}"],
    "Genes_Involved": ["-", ", ".join(tp_genes), ", ".join(fp_genes), ", ".join(fn_genes), "Remaining dataset", "-", "-", "-", "-"]
}

summary_df = pd.DataFrame(summary_data)
csv_output_path = os.path.join(OUTPUT_DIR, "Ribosomal_detection_summary.csv")


summary_df.to_csv(csv_output_path, index=False)

print(f"Evaluation complete.")
print(f"Summary saved to: {csv_output_path}")

# Quick print to console for immediate checking
print("\n--- Quick Stats ---")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")

In [ ]:
#"dnaE", "dnaA", "dnaC", "dnaN", "dnaX", "holB", "holA"

import pandas as pd
import os

# --- 1. INPUTS AND SETUP ---
# Update these paths to match your local setup
PROJECT_ROOT = r'E:\Thesis3april'
anno_path = os.path.join(PROJECT_ROOT, "files", "Pathway_annotation.xlsx")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "statistics", "evaluation_results")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Load your annotation data
anno_df = pd.read_excel(anno_path)

# --- 2. DEFINE YOUR DETECTIONS ---
# These are the genes you identified from the plot for the specific category
target_category = "DNA condensation/ segregation"
my_detections = {"dnaE", "dnaA", "dnaC", "dnaN", "dnaX", "holB", "holA"}

# --- 3. CALCULATION LOGIC ---
# Get Ground Truth from the Excel file based on your plotting logic
# We look at both Annotation 4 and 3
ground_truth_mask = (
    (anno_df['SubtiWiki Annotation 4'] == target_category) | 
    ((anno_df['SubtiWiki Annotation 4'].isna()) & (anno_df['SubtiWiki Annotation 3'] == target_category))
)
ground_truth_genes = set(anno_df.loc[ground_truth_mask, 'Treatment'].unique())
all_genes_in_dataset = set(anno_df['Treatment'].unique())

# Core Metrics
tp_genes = my_detections.intersection(ground_truth_genes)
fp_genes = my_detections - ground_truth_genes
fn_genes = ground_truth_genes - my_detections

tp = len(tp_genes)
fp = len(fp_genes)
fn = len(fn_genes)
tn = len(all_genes_in_dataset - (ground_truth_genes | my_detections))

# Math for Metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
accuracy = (tp + tn) / (tp + fp + fn + tn)
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

# --- 4. CREATE SUMMARY AND SAVE ---
summary_data = {
    "Metric": ["Category", "True Positives", "False Positives", "False Negatives", "True Negatives", "Precision", "Recall", "Accuracy", "F1-Score"],
    "Value": [target_category, tp, fp, fn, tn, f"{precision:.4f}", f"{recall:.4f}", f"{accuracy:.4f}", f"{f1:.4f}"],
    "Genes_Involved": ["-", ", ".join(tp_genes), ", ".join(fp_genes), ", ".join(fn_genes), "Remaining dataset", "-", "-", "-", "-"]
}

summary_df = pd.DataFrame(summary_data)
csv_output_path = os.path.join(OUTPUT_DIR, "DNA condensation_segregation_detection_summary.csv")
os.makedirs(csv_output_path, exist_ok=True)


print(f"Evaluation complete.")
print(f"Summary saved to: {csv_output_path}")

# Quick print to console for immediate checking
print("\n--- Quick Stats ---")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")